# Time-Series Forecasting — Walk-Forward CV, Lag Features, and Baselines

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb16_time_series_forecasting_student.ipynb)


> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. Complete both to receive participation credit.

---


## Learning Objectives

By the end of this notebook, you will be able to:

1. Distinguish a forecasting problem from a generic supervised-learning problem and choose the right evaluation protocol.
2. Run a structured time-series EDA — time plot, seasonal sub-series, decomposition, autocorrelation — on a real labor-market dataset.
3. Build a **time-respecting** train/test split where the **12-month locked test window** matches the planner's forecast horizon, and walk-forward CV handles model selection on the training window.
4. Run **walk-forward cross-validation** with `ExpandingWindowSplitter` from `sktime` instead of k-fold CV (which would shuffle time and leak the future into the past).
5. Compare four classical forecasting benchmarks (Mean, Naive, Seasonal-Naive, Drift) against a learned **lag-feature linear regression** on identical CV folds.
6. Add **regularization** (Ridge) to the lag-feature linear model and decide whether it earns its place via the Student's *t* 95% CI overlap rule.
7. Open the locked test window in a **one-shot evaluation ceremony**, mirroring nb14's protocol but adapted to time.


## Setup

Import the libraries we will use across the notebook. Most of the toolkit is familiar from Week 1 — `pandas`, `matplotlib`, `seaborn`, `LinearRegression`, `Ridge`, `mean_absolute_error`, and `scipy.stats.t` for the Student's *t* CIs from nb08. The EDA tools are the same: `STL` and `plot_acf` from `statsmodels`.

What is new today is **`sktime`** — the time-series forecasting framework introduced in the lecture slides. Three `sktime` tools replace the manual code you would otherwise have to write by hand: **`NaiveForecaster`** (the four classical benchmarks in one class), **`make_reduction`** (wraps any sklearn `Pipeline` as a time-respecting forecaster with automatic lag features — the nb02 Pipeline principle applied to forecasting), and **`ExpandingWindowSplitter`** (walk-forward CV that replaces `TimeSeriesSplit`). The `sktime` API mirrors sklearn's `.fit()` / `.predict()` interface, so the vocabulary is familiar — only the time-respecting plumbing underneath is new.

In [ ]:
# Setup Cell — install sktime (not pre-installed in Colab)
!pip install -q sktime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf

# sktime — time-series forecasting framework (lecture 08 vocabulary)
from sktime.forecasting.naive import NaiveForecaster
from sktime.forecasting.compose import make_reduction
from sktime.split import temporal_train_test_split, ExpandingWindowSplitter

# sklearn — the estimators that sktime wraps via make_reduction
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import mean_absolute_error
from scipy.stats import t as student_t

warnings.filterwarnings("ignore")

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.precision', 3)
sns.set_style("whitegrid")

print("Setup complete!")
print(f"RANDOM_SEED = {RANDOM_SEED}")

**Reading the output:** A clean `Setup complete!` confirms all libraries loaded without errors. The `!pip install -q sktime` line installs `sktime` in Colab (it is not pre-installed). If the install fails, try `Runtime → Restart runtime` and re-run the cell. The `RANDOM_SEED = 474` and `figure.figsize = (10, 6)` settings match every prior notebook, so your outputs will reproduce exactly.

---

## 1. Why This Matters: Forecasting US Retail Employment

The **US Bureau of Labor Statistics** publishes monthly employment counts for every major industry. A **state-level workforce planner** uses those numbers to forecast next year's retail labor demand:

> *"I need a defensible forecast of US retail-sector employment one year out — with a confidence interval the legislature can read. Last year's headline number is not enough; I need the model and the diagnostics."*

This is **not** the kind of problem we solved in nb01–nb15. There, every row was an independent observation and a 60/20/20 random split was the right protocol. Here, the rows are months in a sequence — the **order matters**, and shuffling them would let the model peek at the future during training (a classic data leak). The fix is structural: the test window is always the **most recent** slice of history, and cross-validation walks forward in time.

This notebook ports the **Week-1 analytics workflow** (EDA → split → baselines → linear features → regularization) to the time-series setting, threading the same structural rule throughout: *the future cannot leak into the past.*

**A question that often comes up here:** *"Why isn't this just nb14 with a different metric?"* Two reasons. First, k-fold CV with shuffled rows would let row 50 be in the training fold and row 49 in the validation fold — the model would see a future month while learning to predict an earlier one. That defeats the entire idea of forecasting. Second, employment series have **seasonality** (holiday hiring) and **long-run trend** (decades of structural growth), so a feature engineered as "value 12 months ago" is structurally meaningful in a way that "row 12 in the dataset" is not.


## 2. Load the Data and Sanity Checks

We use the **US Employment dataset** from *Forecasting: Principles and Practice* (FPP3), the standard open-source textbook for time-series analysis. The dataset contains monthly employment counts (in thousands) for every major BLS industry classification from 1939 onward — roughly 80 years of monthly observations across dozens of industries. We filter to the **"Retail Trade"** series because it is the workforce planner's target and because it has all three structural features that make forecasting interesting: long-run trend, clear annual seasonality, and occasional recession-driven disruptions. The `ds` column is the date stamp; `y` is the employment count.

> 💡 **Gemini Prompt:** *"Read the US Employment CSV from this raw GitHub URL: `https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/lecture_slides/08_time_series/data/us_employment.csv` using `pd.read_csv` with `parse_dates=["ds"]`. Filter the DataFrame to `unique_id == "Retail Trade"`, keep only the `ds` and `y` columns, sort by `ds`, and reset the index. Then convert to a pandas Series `y` indexed by date, and set the index to a monthly PeriodIndex via `y.index.to_period("M")`. Print the row count, the date range from `y.index[0]` to `y.index[-1]`, the count of missing values, and finally display `df.head()`."*
>
> **After running, verify:**
> - [ ] Row count is approximately 960 monthly observations
> - [ ] Date range spans 1939-01 through 2019-09
> - [ ] Missing values equals 0
> - [ ] `y` is a pandas Series with PeriodIndex (freq=`M`)
> - [ ] `df.head()` shows ds (Timestamp) and y (employment) columns

In [ ]:
# Load the full dataset directly from the course's GitHub raw URL
DATA_URL = (
    "https://raw.githubusercontent.com/davi-moreira/"
    "2026Summer_predictive_analytics_purdue_MGMT474/main/"
    "lecture_slides/08_time_series/data/us_employment.csv"
)
us_employment = pd.read_csv(DATA_URL, parse_dates=["ds"])

# Filter to the Retail Trade series
df = (
    us_employment.query('unique_id == "Retail Trade"')
    .loc[:, ["ds", "y"]]
    .sort_values("ds")
    .reset_index(drop=True)
)

# Convert to sktime-compatible format: a pandas Series with a PeriodIndex.
# sktime forecasters expect a time-indexed Series, not a DataFrame.
y = df.set_index("ds")["y"]
y.index = y.index.to_period("M")

print(f"Rows: {len(y):,}")
print(f"Date range: {y.index[0]}  ->  {y.index[-1]}")
print(f"Missing values: {y.isna().sum()}")
print()
df.head()

**Reading the output:**

For the workforce planner, this dataset is the foundation of the forecast she will present to the legislature. You should see roughly **960 rows** spanning **1939-01 through 2019-09** (80 years of monthly data) with **zero missing values**. The two columns are `ds` (date stamp) and `y` (employment in thousands). After loading, we convert the DataFrame to a **pandas Series with a PeriodIndex** (`y.index.to_period("M")`) — this is the format `sktime` forecasters expect. The 80-year history gives the planner enough past cycles to learn the seasonal pattern and enough structural breaks (recessions) to calibrate her prediction intervals. The DataFrame `df` stays around for the EDA plots in §3, which use `df["ds"]` and `df["y"]` directly.

> **A question that often comes up here:** *"Why is `unique_id` a string column?"* The original dataset is in long format — one row per (industry × month). The `unique_id` column tags each row with its industry. We filter to a single one so the analysis stays focused.

**The planner's conclusion:** The dataset is fit for purpose — 80 years of clean monthly retail-employment data, zero missing values, no anomalies. I have enough history to identify long-run trend, the annual seasonal cycle, and recession-driven shocks. I move directly to the visual EDA in §3 without any cleaning detours.

## 3. Time-Series EDA — Six Plots, One Story

Before the workforce planner builds any model, she needs to understand the structure of the data she is forecasting. A time series asks for visual EDA before any modeling — the plots below answer the specific questions the legislature will ask: *"Is employment trending up or down? Is there a holiday hiring cycle? How big are recession-driven disruptions?"* The canonical sequence is: **time plot** (the whole series), **time plot zoomed** (a recent slice), **seasonal sub-series** (per-month box plot), **STL decomposition** (trend + seasonal + remainder), **ACF** (autocorrelation function), **lag-1 scatter** (do consecutive months track each other?). Six plots, one combined story.


### 3.1 Time plot — the whole series

The first plot every forecaster makes. Plot the full series on a timeline and let the shape speak: an upward or downward drift means **trend**, regular ripples mean **seasonality**, and sharp discontinuities mean **structural breaks** (recessions, policy changes). No statistical test conveys these features as quickly as a single well-drawn time plot — and the workforce planner who skips this step risks building a model that ignores structure the eye catches in seconds.

> 💡 **Gemini Prompt:** *"Create a `matplotlib` figure with `plt.subplots(figsize=(12, 5))`. Plot `df["ds"]` on the x-axis and `df["y"]` on the y-axis as a single line using `color="#1f77b4"` and `linewidth=0.8`. Label the x-axis "Year", the y-axis "Employment (thousands)", set the title to "US Retail Trade Employment, 1939\u20132019 (monthly)", call `plt.tight_layout()` and `plt.show()`."*
>
> **After running, verify:**
> - [ ] One blue line spanning 1939 through 2019
> - [ ] Employment grows from roughly 5,000 to roughly 16,000 thousand
> - [ ] The 2008 recession dip is visible near the right side
> - [ ] Small annual ripples are visible (seasonality)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df["ds"], df["y"], color="#1f77b4", linewidth=0.8)
ax.set_xlabel("Year")
ax.set_ylabel("Employment (thousands)")
ax.set_title("US Retail Trade Employment, 1939–2019 (monthly)")
plt.tight_layout()
plt.show()


**Reading the output:**

Three structural components are visible in this single plot, and the workforce planner needs to name all three before modeling begins.

**1. Long-run trend.** Employment roughly **triples** from about 5,000 thousand in 1939 to over 15,000 thousand by 2019. The growth is not uniform — you can trace the mid-century post-war expansion, the 1990s retail and e-commerce boom, and the gradual recovery after 2010. The overall shape is an upward curve that any forecasting model must track; a model that ignores it would predict the 1950s level for 2020.

**2. Seasonality.** Look closely and you will see small annual ripples running along the trend line — the curve is not smooth but gently serrated. Those ripples are **holiday retail hiring** in November and December (the peaks) and **post-holiday layoffs** in January and February (the troughs). At this 80-year zoom level the ripples are hard to read, which is exactly why the next plot zooms in. But even here, the regularity is visible: the same up-down pattern repeats every 12 months for eight decades. A model that captures trend but ignores seasonality will systematically over-predict in January and under-predict in December.

**3. Structural breaks.** The sharpest disruptions are **recessions**: the 2008–2010 financial crisis is the most dramatic (a steep drop of roughly 2,000 thousand employees followed by a multi-year recovery), but smaller dips are visible in 1974 (oil crisis), 1980 (double-dip recession), 1990, and 2001 (dot-com bust). These breaks are driven by macroeconomic shocks that the series itself cannot predict — no lag feature or seasonal pattern will warn you that a financial crisis is coming. The practical implication for the workforce planner: the model's prediction interval must be wide enough to cover these tail events, and any forecast delivered to the legislature should carry a caveat about recession-driven uncertainty.

A single forecasting model has to capture trend and seasonality (the learnable components) while honestly acknowledging that structural breaks will occasionally push actuals outside even a well-calibrated prediction interval.

**The planner's conclusion:** The series has three structural components I must address — long-run trend (the model needs lag-1 or another trend carrier), annual seasonality (the model needs a 12-month anchor), and recession-driven shocks (the model cannot predict them, so my prediction interval must be wide enough to absorb them). I move to §3.2 to read the seasonal amplitude more clearly at decade-scale zoom.

### 3.2 Time plot zoomed — last 10 years

The full 80-year time plot compresses 960 monthly observations into a single curve, which makes the annual seasonal cycle nearly invisible — the ripples are too small relative to the 80-year trend. Zooming in to the last decade stretches the x-axis enough to see individual holiday peaks and post-holiday dips. This is the plot that tells the workforce planner *how much* staffing swings within a single year.

> 💡 **Gemini Prompt:** *"Filter `df` to rows where `df["ds"] >= "2010-01-01"` and store as `recent`. Create a figure with `figsize=(12, 5)`. Plot `recent["ds"]` vs `recent["y"]` with markers and a line using format `"o-"`, `color="#2ca02c"`, `markersize=4`. Set the title to "US Retail Trade Employment \u2014 2010\u20132019 (monthly)", label the axes "Year" and "Employment (thousands)", tight_layout, show."*
>
> **After running, verify:**
> - [ ] Green line with markers covering 2010 through 2019
> - [ ] Annual November/December peaks are clearly visible
> - [ ] January/February troughs are clearly visible
> - [ ] An overall upward trend across the decade

In [ ]:
recent = df[df["ds"] >= "2010-01-01"]
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(recent["ds"], recent["y"], "o-", color="#2ca02c", markersize=4)
ax.set_title("US Retail Trade Employment — 2010–2019 (monthly)")
ax.set_xlabel("Year")
ax.set_ylabel("Employment (thousands)")
plt.tight_layout()
plt.show()


**Reading the output:**

The zoomed plot makes the seasonal cycle unmistakable. Three features are now readable that the 80-year plot compressed into illegibility.

**Annual peaks in November/December.** Every year, employment surges as retailers staff up for the holiday season — Black Friday through Christmas. The peaks are the tallest points in each annual wave. For the workforce planner, these peaks are not surprises; they are predictable staffing events that drive temporary-hire budgets, training timelines, and warehouse capacity decisions months in advance.

**Post-holiday troughs in January/February.** After the holidays, seasonal positions end and employment drops sharply. The trough is typically the lowest point of the annual cycle. The gap between the December peak and the January trough — a swing of several percent of the total retail workforce — is the seasonal amplitude the model needs to capture.

**Slow upward trend underneath the waves.** Each year's trough is slightly higher than the previous year's trough; each peak is slightly higher than the previous peak. That is the long-run trend visible at this scale — the same structural growth that §3.1's full time plot showed over 80 years, now legible as a gentle upward tilt beneath the seasonal ripples.

This seasonal cycle is exactly the structure that a **lag-12 feature** (this month vs. the same month last year) will capture automatically in section 8. The model does not need to "know" about holiday hiring — it just needs to see that December 2018 looked a lot like December 2017.

**The planner's conclusion:** The seasonal swing is operationally material — a holiday peak in December, a January trough, with several percent of the workforce moving between them. The legislature staffs around this cycle every year, so the forecast must capture it. A **lag-12 feature** (this month vs. the same month last year) will let the model learn the cycle automatically. I proceed to §3.3 to verify the seasonal shape is stable across decades — not a recent artifact.

### 3.3 Seasonal sub-series box plot

A box plot of `y` grouped by **month-of-year** compresses 80 Decembers into one box, 80 Januaries into another, and so on. It answers two questions at once: *"how strong is the seasonal pattern?"* (do the medians differ across months?) and *"how stable is it?"* (are the boxes tight or sprawling?). If December's box sits consistently above June's across 80 years, the seasonal effect is real and worth modeling. If every month's box overlaps every other, there is no seasonality to capture.

> 💡 **Gemini Prompt:** *"Copy `df` to `df_seasonal` and add a column `"month"` equal to `df_seasonal["ds"].dt.month`. Create a figure with `figsize=(11, 5)`. Use `sns.boxplot` with `data=df_seasonal`, `x="month"`, `y="y"`, and `color="#ff7f0e"`. Set the title to "Seasonal sub-series \u2014 employment distribution by month-of-year", label axes "Month of year" and "Employment (thousands)", tight_layout, show."*
>
> **After running, verify:**
> - [ ] Twelve orange boxes on the x-axis (months 1\u201312)
> - [ ] December (month 12) has the highest median
> - [ ] January (month 1) has the lowest median
> - [ ] Boxes are wide (the trend stretches each box vertically)

In [ ]:
df_seasonal = df.copy()
df_seasonal["month"] = df_seasonal["ds"].dt.month
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df_seasonal, x="month", y="y", ax=ax, color="#ff7f0e")
ax.set_title("Seasonal sub-series — employment distribution by month-of-year")
ax.set_xlabel("Month of year")
ax.set_ylabel("Employment (thousands)")
plt.tight_layout()
plt.show()


**Reading the output:**

The box plot answers both seasonal questions at once.

**How strong is the seasonal pattern?** Strong enough to act on. **December** sits visibly above every other month — this is the holiday retail hiring surge at its clearest. November comes next (early holiday ramp-up and pre-Black Friday staffing), then a steady plateau from March through October, then the **January/February dip** as seasonal positions end and post-holiday returns wind down. The medians trace a smooth annual cycle that the workforce planner can use as a staffing calendar: expect the highest labor demand in December, the lowest in January, and a stable mid-range from spring through early fall.

**How stable is the seasonal pattern?** Stable in shape, but the boxes are wide. That width is not noise — it is the **long-run trend hiding inside the by-month aggregation**. Think about what a single "December" box contains: December 1945 (roughly 6,000 thousand employees) and December 2015 (roughly 16,000 thousand). Both are Decembers, but they sit at very different absolute levels because the economy grew over those 70 years. When you lump them into one box, the box stretches from 6,000 to 16,000 — a range driven by the trend, not by December-to-December instability. The seasonal *shape* (December > November > ... > January) is real and consistent across decades; the seasonal *amplitude* relative to the trend is moderate.

This distinction — real seasonal shape but trend-inflated box width — is why the STL decomposition in the next plot is valuable. STL separates the trend from the seasonal component mathematically, so you can see each one's contribution cleanly.

**The planner's conclusion:** The seasonal *shape* is stable across 80 years — December always sits at the top, January at the bottom, the same monthly pattern repeating. The boxes are wide because of the long-run trend, not because the seasonality is unreliable. I can confidently model the cycle with `lag12`. I now move to §3.4's STL decomposition to separate trend from seasonal mathematically and to see the residual structure — that residual will tell me how wide my prediction interval needs to be.

### 3.4 STL decomposition — trend + seasonal + remainder

#### Time Series Components

Every business time series can be expressed as a combination of underlying components, each representing a different type of structure. Naming these components is the first step before any forecasting: it tells the workforce planner which pieces of the series are predictable, which repeat on a calendar, and which are noise the model cannot anticipate. Two structural forms are common in practice — **additive** and **multiplicative**.

#### Additive Structure

$$y_t = S_t + T_t + R_t$$

where:

- $y_t$ = **observed value** at time $t$ (the actual monthly employment count, the raw data you have been plotting),
- $S_t$ = **seasonal component** — the regular, repeating calendar-driven pattern (December peaks, January troughs in retail),
- $T_t$ = **trend-cycle component** — the smooth long-run trajectory after the seasonal ups and downs are stripped away (the answer to *"ignoring the holiday cycle, where is employment heading?"*),
- $R_t$ = **remainder** (or residual) — everything the trend and seasonal components cannot explain. Small values mean the decomposition captured most of the structure; large spikes mean a shock occurred (a recession, a policy change) that neither pattern anticipated.

The additive model is appropriate when the **amplitude of seasonal and random variations remains roughly constant** regardless of the level of the series. In plain English: if December's holiday hiring surge is *roughly the same size in employees* whether the workforce sits at 5 million (1950) or 16 million (2015), the seasonal swing is additive.

#### Multiplicative Structure

$$y_t = S_t \times T_t \times R_t$$

The multiplicative model is suitable when **seasonal or irregular variations increase or decrease proportionally with the level of the series** — a pattern commonly observed in fast-growing economic data. Concrete example: if December's holiday surge is consistently *10 percent above the average level*, then the swing grows in absolute size as the workforce grows (10% of 5 million is 500 thousand; 10% of 16 million is 1.6 million). The seasonal *amplitude* tracks the *level*.

#### Log Transformation Equivalence

A multiplicative relationship can be converted into an additive one by applying a logarithm:

$$y_t = S_t \times T_t \times R_t \Longleftrightarrow \log y_t = \log S_t + \log T_t + \log R_t$$

Stabilizing variance with a log transformation lets us use **additive decomposition tools** (like STL) even for series with proportional variability. This is a common pre-processing step for economic and financial series whose seasonal swings grow with the level.

#### Which one applies to US Retail Trade?

Eyeball test: does the seasonal wave in §3.2's zoomed time plot look the same height in 1950 (low employment) and in 2015 (high employment)? If yes → **additive**. If December's swing is visibly bigger in absolute employees when overall employment is higher → **multiplicative** (or use a log transformation first). For US Retail Trade, the seasonal swing is roughly constant in absolute size across decades, so we use **additive STL**.

#### STL — what it does

**STL (Seasonal-Trend decomposition using Loess)** implements the additive decomposition above. Given a series with a known seasonal period (12 for monthly data), STL fits a smooth trend with a Loess regression, isolates a stable seasonal pattern, and assigns whatever is left to the remainder. The four panels you see in the next plot are the raw data, the trend, the seasonal component, and the remainder — exactly the four pieces of the additive equation, drawn separately so the workforce planner can read each one independently.


> 💡 **Gemini Prompt:** *"Build a Series `ts` from `df` by setting `ds` as the index and keeping `y`. Run `STL(ts, period=12, robust=True).fit()` and store as `stl`. Create a 4\u00d71 subplot with `figsize=(12, 9)` and `sharex=True`. In `axes[0]` plot `ts` in black with ylabel "y (data)". In `axes[1]` plot `stl.trend` in `"#1f77b4"` with ylabel "Trend". In `axes[2]` plot `stl.seasonal` in `"#2ca02c"` with ylabel "Seasonal". In `axes[3]` plot `stl.resid` in `"#d62728"` with ylabel "Remainder" and add `axhline(0, color="black", linewidth=0.5)`. Set `axes[0]` title to "STL Decomposition \u2014 US Retail Trade Employment". tight_layout, show."*
>
> **After running, verify:**
> - [ ] Four vertically stacked panels sharing the same x-axis
> - [ ] Trend panel is smooth and rising; 2008 dip is visible
> - [ ] Seasonal panel shows a regular wave repeating every 12 months
> - [ ] Remainder panel has spikes near recessions (1974, 2008)

In [ ]:
# STL decomposition with monthly period
ts = df.set_index("ds")["y"]
stl = STL(ts, period=12, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
axes[0].plot(ts.index, ts.values, color="black"); axes[0].set_ylabel("y (data)")
axes[1].plot(ts.index, stl.trend, color="#1f77b4"); axes[1].set_ylabel("Trend")
axes[2].plot(ts.index, stl.seasonal, color="#2ca02c"); axes[2].set_ylabel("Seasonal")
axes[3].plot(ts.index, stl.resid, color="#d62728"); axes[3].set_ylabel("Remainder")
axes[3].axhline(0, color="black", linewidth=0.5)
axes[0].set_title("STL Decomposition — US Retail Trade Employment")
plt.tight_layout()
plt.show()


**Reading the output:**

Four panels, top to bottom, each isolating one component of the series.

**Data (top panel).** The raw series — the same curve you saw in §3.1's time plot. It contains all three structural components mixed together.

**Trend (second panel).** The smooth long-run component after the seasonal and residual fluctuations have been stripped away. You can now see the structural growth trajectory clearly: steady expansion from the 1940s through the 1970s, a plateau and mild dips around the oil-crisis recessions, accelerating growth through the 1990s retail boom, the sharp **2008 financial-crisis drop** (roughly 2,000 thousand employees lost in two years), and the gradual post-2010 recovery. This is the component that lag-1 features will capture — each month's employment level is closely related to last month's.

**Seasonal (third panel).** The regular annual pattern, isolated from the trend. The same wave shape repeats every 12 months for 80 years — December peaks, January troughs, a consistent amplitude throughout. Notice that the wave height does not grow over time even though the trend does; this confirms the **additive** decomposition was the right choice. If the seasonal swings had grown proportionally with the level (bigger absolute swings at higher employment), a multiplicative decomposition would have been needed instead.

**Remainder (bottom panel).** What neither trend nor seasonality explains. Most of the time, the remainder fluctuates in a narrow band around zero — the trend and seasonal components account for nearly all of the variation. But there are visible **spikes around recessions**: the 2008 crisis produces the largest residual, and smaller spikes appear in 1974, 1980, 1990, and 2001. These are the structural breaks from §3.1 — macroeconomic shocks that arrive from outside the series. No lag feature or seasonal pattern will predict them. For the workforce planner, the remainder panel is a warning: the prediction interval she quotes to the legislature must be wide enough to accommodate recession-sized shocks, even though the model cannot see them coming.

> **A question that often comes up here:** *"Should I use additive or multiplicative decomposition?"* Additive when the seasonal amplitude does not grow with the trend; multiplicative when it does. Visually, the seasonal swing here is roughly the same height in 1950 (small absolute employment) as in 2010 (large absolute employment) → additive is the right call. If the seasonal swing was larger in absolute terms when employment was higher, multiplicative would fit better.

**The planner's conclusion:** The decomposition confirms an **additive** structure — trend + seasonal + remainder — with a stable seasonal amplitude across the entire 80-year history. The remainder panel reveals the recession spikes I cannot model from the series alone (2008 is the largest, with smaller spikes in 1974, 1980, 1990, 2001). For the legislative report I will note explicitly: *the prediction interval is calibrated against ordinary monthly variation; recession-scale shocks may push actuals outside the band*. Next, §3.5's ACF picks the specific lags I will engineer as features.

### 3.5 Autocorrelation (ACF) plot

The ACF asks *"how strongly does month $t$'s value depend on month $t-k$'s value, for each lag $k$?"*. Formally, the autocorrelation at lag $k$ is:

$$r_k = \frac{\sum_{t=k+1}^{n}(y_t - \bar{y})(y_{t-k} - \bar{y})}{\sum_{t=1}^{n}(y_t - \bar{y})^2}$$

Here is what each piece means:

- $y_t$ is the employment value in month $t$ (the current month you are looking at).
- $y_{t-k}$ is the employment value $k$ months earlier — the "lagged" value. When $k = 1$, it is last month; when $k = 12$, it is the same month one year ago.
- $\bar{y}$ is the overall mean of the series — the average employment across all 960 months.
- The numerator measures how much month $t$ and month $t-k$ move together *relative to the mean*. If both are above average at the same time (or both below), the product is positive and the correlation is strong.
- The denominator is the total variance of the series — it scales the numerator so $r_k$ always falls between $-1$ and $+1$.

In practice, you do not compute this by hand — `plot_acf` does it for you. What matters is reading the bar chart: each bar is one $r_k$ value. Tall bars at lags 1, 2, 3 mean **trend and momentum** — recent months are highly correlated with the present. A tall bar at lag 12 (and again at 24) means **annual seasonality** — what happened a year ago is a strong predictor of what happens now. The ACF is the empirical tool that tells the workforce planner *which pieces of history are worth building into the forecast model*. Instead of guessing which lags to include, the planner lets the data answer the question directly.

> 💡 **Gemini Prompt:** *"Create a figure with `figsize=(11, 4)`. Call `plot_acf(ts.values, lags=36, ax=ax, zero=False)` to draw the autocorrelation function for lags 1 through 36. Set the title to "Autocorrelation function \u2014 lags 1 through 36", label the x-axis "Lag (months)", tight_layout, show."*
>
> **After running, verify:**
> - [ ] Bars decay slowly from lag 1 through lag 24 (trend signature)
> - [ ] Visible local peaks at lag 12 and lag 24 (seasonality)
> - [ ] All bars in the visible range exceed the blue significance band

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
plot_acf(ts.values, lags=36, ax=ax, zero=False)
ax.set_title("Autocorrelation function — lags 1 through 36")
ax.set_xlabel("Lag (months)")
plt.tight_layout()
plt.show()


**Reading the output:**

The ACF translates the visual patterns from the first four plots into precise numerical evidence about which lags carry predictive signal.

**Slow decay across lags 1–24.** The bars start tall at lag 1 (correlation above 0.95) and decline gradually through lag 24. This slow decay is the **signature of a strong trend** — when employment is high this month, it was also high last month, and the month before that, and so on. The practical implication: the most recent value (`lag1`) is by far the single most informative predictor of the next value. This is why the naive forecast ("next month = last month") will be so hard to beat.

**Local peaks at lags 12 and 24.** On top of the slow decay, the bars at lags 12 and 24 are visibly taller than their neighbors. Lag 12 means "this month correlates strongly with the same month one year ago" — that is the annual seasonal cycle the zoomed time plot and the box plot already showed. Lag 24 means the same pattern holds two years back. These peaks are weaker than lag 1 (the trend dominates), but they carry **independent seasonal information** that lag 1 alone cannot provide.

**The blue shaded band** marks the 95% confidence interval for "no significant autocorrelation." Every bar that extends beyond the band is statistically significant. On this series, every bar through lag 36 is significant — the series has strong, persistent structure at every timescale up to three years.

This ACF gives us **direct empirical justification** for the two lag features we will engineer in section 8: `lag1` captures the trend and short-term momentum (the slow decay), and `lag12` captures the annual seasonal cycle (the lag-12 peak). The ACF is the diagnostic; the features are the response. When the planner presents the forecast to the legislature, she can point to this plot and say: *"The model uses last month's employment and the same month last year — these are the two signals the data itself told us to use."*

**The planner's conclusion:** Lag-1 and lag-12 are the two empirically justified features — the data itself told me which signals to use, no guessing required. I will engineer exactly these two lags in the model (the slow decay between them adds little independent information, and parsimony is a virtue when defending the model to the legislature). Before I commit to modeling, §3.6 verifies one more sanity check: how good is the simplest possible forecast — "next month equals last month" — going to be?

### 3.6 Lag-1 scatter

The simplest forecast in the world is *"next month equals last month."* The lag-1 scatter plots this month's employment against last month's — every dot is one month. If the dots hug the 45° line, the naive forecast is strong (month-to-month changes are small). If they scatter into a cloud, consecutive months are volatile and the naive forecast will have large errors. For the workforce planner, this plot answers a practical question: *"can I get away with just using last month's number, or do I genuinely need a model?"*

> 💡 **Gemini Prompt:** *"Build `lag1_df` by assigning a column `lag1 = df["y"].shift(1)` then dropping NaNs. Create a figure with `figsize=(7, 7)`. Scatter `lag1_df["lag1"]` (x) against `lag1_df["y"]` (y) with `s=8`, `alpha=0.5`, `color="#9467bd"`. Compute `lo = lag1_df["y"].min()` and `hi = lag1_df["y"].max()`, then plot a dashed reference line from `(lo, lo)` to `(hi, hi)` with `"k--"`, `linewidth=0.8`, and label "Perfect lag-1 forecast (y = lag1)". Label axes "Employment, t-1 (lag1)" and "Employment, t", set a title that asks whether the previous month predicts the current month, add legend, tight_layout, show."*
>
> **After running, verify:**
> - [ ] Purple scatter points hug the diagonal dashed line tightly
> - [ ] Cloud is elongated (low-left to upper-right) due to trend
> - [ ] No dramatic outliers far from the diagonal

In [ ]:
lag1_df = df.assign(lag1=df["y"].shift(1)).dropna()
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(lag1_df["lag1"], lag1_df["y"], s=8, alpha=0.5, color="#9467bd")
lo, hi = lag1_df["y"].min(), lag1_df["y"].max()
ax.plot([lo, hi], [lo, hi], "k--", linewidth=0.8, label="Perfect lag-1 forecast (y = lag1)")
ax.set_xlabel("Employment, t-1 (lag1)")
ax.set_ylabel("Employment, t")
ax.set_title("Lag-1 scatter — does the previous month predict the current month?")
ax.legend()
plt.tight_layout()
plt.show()


**Reading the output:**

Every dot in this scatter is one month. The x-coordinate is last month's employment; the y-coordinate is this month's. The dashed diagonal is the 45° line — the line where "this month = last month" exactly.

**The dots hug the 45° line tightly.** Month-to-month changes in retail employment are small relative to the overall level. A month at 15,000 thousand employees is almost always followed by a month between 14,800 and 15,200 — a change of at most 1–2%. That tightness is the visual proof that the **naive forecast** ("next month = last month") will be a strong baseline. Any learned model that does not beat it is not earning its keep.

**The cloud is elongated, not round.** The scatter stretches from the lower-left (early decades, lower employment) to the upper-right (recent decades, higher employment). That elongation is the trend — the series moves through different employment regimes over 80 years. Within each regime, the dots cluster tightly around the diagonal.

**There are no dramatic outliers far from the line.** Even the recession months (2008–2009) do not produce dots that land far off the diagonal, because the employment drops happened over multiple months rather than in a single catastrophic jump. This is good news for the workforce planner's lag-feature regression: the relationship between consecutive months is approximately linear and stable. The planner can confidently tell the legislature that a simple linear model captures the month-to-month dynamics — no need for complex nonlinear machinery.

> **A question that often comes up here:** *"Does this mean the naive forecast will always be hard to beat?"* For slow-moving, strongly autocorrelated series like monthly employment, yes — the naive forecast inherits most of the signal for free. But for volatile series — daily tech stocks, hourly web traffic, cryptocurrency — the lag-1 scatter would show a much wider cloud, and the naive baseline would have much larger errors. The tighter the scatter hugs the 45° line, the higher the bar any learned model has to clear.

Six plots, one combined story: strong trend, clear annual seasonality, and tight lag-1 autocorrelation. Section 4 translates those structural facts into the single rule that governs every modeling decision below.

**The planner's conclusion:** The lag-1 scatter sets the bar: the naive forecast (next month = last month) will be tough to beat because month-to-month change is small relative to the level. My learned model must clear that bar with non-overlapping CIs in §8, or I have no justification for adding model complexity. The EDA is now complete. I have my feature set — `[lag1, lag12]` — and I move to §4's structural rule, then to §5's split.

---

## 4. The Structural Rule — Never Shuffle, Never Leak the Future

Every static-classification rule we built since nb01 still works in this notebook — **except one**. Rows here are months in a sequence, and shuffling them would let the model peek at the future during training. That single structural change cascades into three downstream changes:

1. **Train/test split**: the test window is the **most recent slice** of history, not a random sample.
2. **Cross-validation**: every fold's training data must come strictly **before** its validation data.
3. **Features**: lag features (last month, 12 months ago) replace random feature engineering.

Sections 5–7 implement each one in order. For the workforce planner, respecting the arrow of time is not just a statistical requirement — it is a credibility requirement. A forecast model validated on data from before the training period would not survive scrutiny in a legislative hearing.

---


## 5. Time-Respecting Split — 12-Month Locked Test Window

The workforce planner's deliverable is a **12-month forecast** for the legislature. The test window should match that horizon exactly: the most recent 12 months of history are **locked** for the one-shot ceremony in §9, and everything before them is the training window.

Why 12 months and not 20%? Three reasons, all grounded in the business case:

1. **The forecast horizon is 12 months.** The planner asks for a one-year forecast — so the test window should evaluate exactly that: can the model predict the next 12 months from a single cutoff?
2. **The most recent history is the most valuable.** Locking 20% (194 months) would hide 16 years of recent dynamics from the model. Locking only 12 months keeps 957 months of history available for training and CV.
3. **The EDA justifies it.** The ACF showed that lag-12 (one year back) is the dominant seasonal feature. A 12-month test window evaluates exactly one complete seasonal cycle — the minimum needed to check whether the model captures the holiday peaks and January troughs.

`temporal_train_test_split` with `test_size=12` carves off the last 12 observations (months) as the locked test window.

> 💡 **Gemini Prompt:** *"Use sktime's `temporal_train_test_split` to split `y` with `test_size=12` (an integer — the most recent 12 observations). Unpack into `y_train, y_test`. Print the train and test date ranges and sample sizes; tag the test line with "[LOCKED \u2014 12 months]". Plot both segments on a figure with `figsize=(12, 4)`: train in `"#1f77b4"`, test in `"#d62728"` with `linestyle="--"`, using `.index.to_timestamp()` on the x-axis. Add a grey dotted vertical line at the train/test boundary. Title: "Time-Respecting Split \u2014 12-Month Locked Test Window". Add legend, tight_layout, show."*
>
> **After running, verify:**
> - [ ] Train spans 1939-01 through 2018-09 (n=957)
> - [ ] Test spans 2018-10 through 2019-09 (n=12, LOCKED)
> - [ ] Plot shows blue train, dashed red test, with a boundary line
> - [ ] No shuffling — chronological order preserved

In [ ]:
# Time-respecting split: train on all history except the last 12 months.
# The locked test window matches the planner's forecast horizon exactly.
y_train, y_test = temporal_train_test_split(y, test_size=12)

print(f"Train: {y_train.index[0]} -> {y_train.index[-1]}  (n={len(y_train)})")
print(f"Test : {y_test.index[0]} -> {y_test.index[-1]}  (n={len(y_test)})  [LOCKED — 12 months]")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y_train.index.to_timestamp(), y_train.values, color="#1f77b4", label="Train")
ax.plot(y_test.index.to_timestamp(), y_test.values, color="#d62728",
        linestyle="--", label="Test (12 months, locked)")
ax.axvline(y_train.index[-1].to_timestamp(), color="grey", linestyle=":", alpha=0.7)
ax.set_title("Time-Respecting Split — 12-Month Locked Test Window")
ax.legend()
plt.tight_layout()
plt.show()

**Reading the output:**

The plot shows the full series divided into two segments.

**Blue (Train)** covers 1939 through September 2018 — 957 months of history. This is the data used for all model fitting and all cross-validation. With 957 months, the model sees nearly 80 years of trend, seasonal cycles, and structural breaks — far more history than a 60% or 80% split would provide.

**Dashed red (Test, 12 months, LOCKED)** covers October 2018 through September 2019 — exactly one year, exactly the forecast horizon the workforce planner delivers to the legislature. This window includes one complete seasonal cycle: a holiday peak, a January trough, and the spring-to-fall plateau. It stays sealed until the §9 ceremony.

> **A question that often comes up here:** *"Is 12 months enough to evaluate the model?"* For this business case, yes — the planner's deliverable IS a 12-month forecast. Testing on 12 months evaluates exactly what the planner will deploy. A longer test window (say, 194 months) would evaluate something the planner never actually does — forecasting 16 years ahead from a single cutoff. The CV comparison in §8, which runs on the 957-month training window, is where the model earns its statistical credibility; the 12-month test is where it earns its operational credibility.

> **A question that often comes up at this point:** *"Should the planner use all 80 years, or only the last 20–30 years?"* It depends on how stable the underlying dynamics are. US retail employment has undergone structural changes since 1939 — the post-war boom, the shift to service-sector dominance, the rise of e-commerce. A planner who believes only recent decades are relevant could restrict the training window to, say, 1990 onward (∼340 months). The trade-off: a shorter window captures more current dynamics but gives the model fewer seasonal cycles to learn from and fewer recessions to calibrate the prediction interval. For this introductory notebook, we use the full history because it maximizes training data and the walk-forward CV design naturally gives more weight to recent folds (they have more history). In production, the planner would test both windows and compare CV MAEs.

**The planner's conclusion:** The split matches my deliverable: 957 months of training (1939 through September 2018), 12 months locked for the final test (October 2018 through September 2019 — exactly the forecast horizon I'll present to the legislature). The training window stays untouched by any model-selection decision; the locked test stays untouched until §9's ceremony. I proceed to §6 to set up walk-forward cross-validation on the training window.

---

## 6. Walk-Forward Cross-Validation with `ExpandingWindowSplitter`

In nb08, `KFold` shuffled rows into training and validation folds — fine when rows are independent. Here, shuffling would let the model see future months while training on earlier ones. `sktime`'s **`ExpandingWindowSplitter`** is the structural fix. It makes three things explicit that sklearn's `TimeSeriesSplit` hides: the **initial training window** (how many months fold 1 trains on), the **step length** (how far the window advances per fold), and the **forecast horizon** (`fh` — how many months ahead each fold predicts). The training window grows with each fold, mimicking real-world deployment where you retrain monthly on an ever-growing history. For the workforce planner, this means each fold simulates a progressively more realistic deployment scenario — early folds forecast with limited history, later folds forecast with decades of accumulated data, which is the situation the planner actually faces today.

> 💡 **Gemini Prompt:** *"Instantiate `cv = ExpandingWindowSplitter(initial_window=int(len(y_train) * 0.5), step_length=int(len(y_train) * 0.1), fh=np.arange(1, 13))`. Get `n_folds = cv.get_n_splits(y_train)` and print a fold calendar with columns Fold, Train start, Train end, Val start, Val end, Train n, Val n — one row per fold, using `y_train.index[tr_idx[0]]` and so on to read the actual PeriodIndex labels. Then create a figure with `figsize=(11, 4)` and visualize each fold as a row: train indices as blue squares (`color="#1f77b4"`, marker="s", `markersize=3`), val indices as orange squares (`color="#ff7f0e"`). Set y-ticks to fold numbers, x-label "Month index in training data", title "Walk-Forward CV: train (blue) always precedes val (orange)", legend, tight_layout, show."*
>
> **After running, verify:**
> - [ ] `n_folds` equals 5
> - [ ] Fold calendar prints actual dates for train/val start/end
> - [ ] Each fold's training window is larger than the previous
> - [ ] Orange squares always sit to the right of blue squares

In [ ]:
# Walk-forward CV with sktime's ExpandingWindowSplitter.
# Three explicit parameters (vs TimeSeriesSplit's implicit defaults):
#   initial_window — how many months the first fold trains on
#   step_length    — how many months the window advances per fold
#   fh             — the forecast horizon (how far ahead each fold predicts)
cv = ExpandingWindowSplitter(
    initial_window=int(len(y_train) * 0.5),
    step_length=int(len(y_train) * 0.1),
    fh=np.arange(1, 13),
)

n_folds = cv.get_n_splits(y_train)

# Fold calendar: train and val date ranges per fold
print(f"Walk-forward CV: {n_folds} folds\n")
print(f"{'Fold':<6} {'Train start':<14} {'Train end':<14} {'Val start':<14} {'Val end':<14} {'Train n':<10} {'Val n'}")
print("-" * 90)
for i, (tr_idx, va_idx) in enumerate(cv.split(y_train)):
    tr_start, tr_end = y_train.index[tr_idx[0]], y_train.index[tr_idx[-1]]
    va_start, va_end = y_train.index[va_idx[0]], y_train.index[va_idx[-1]]
    print(f"{i+1:<6} {str(tr_start):<14} {str(tr_end):<14} {str(va_start):<14} {str(va_end):<14} {len(tr_idx):<10} {len(va_idx)}")

# Visualization
fig, ax = plt.subplots(figsize=(11, 4))
for fold, (tr_idx, va_idx) in enumerate(cv.split(y_train)):
    ax.plot(tr_idx, [fold]*len(tr_idx), "s", color="#1f77b4", markersize=3,
            label="train" if fold == 0 else "")
    ax.plot(va_idx, [fold]*len(va_idx), "s", color="#ff7f0e", markersize=3,
            label="val" if fold == 0 else "")
ax.set_yticks(range(n_folds))
ax.set_yticklabels([f"fold {i+1}" for i in range(n_folds)])
ax.set_xlabel("Month index in training data")
ax.set_title("Walk-Forward CV: train (blue) always precedes val (orange)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Reading the output:**

The visualization shows five rows — one per CV fold — with blue squares marking training months and orange squares marking validation months.

**Fold 1** (top row) has the smallest training window and the first validation window. The model sees only the earliest portion of the training data and is tested on the months that immediately follow. This is the hardest fold — the model has the least history to learn from.

**Fold 5** (bottom row) has the largest training window — roughly five times as much data as fold 1 — and the last validation window. This is the easiest fold and the one closest to what deployment looks like: the model has nearly all the available training history.

Two structural facts to carry forward. First, **orange always sits to the right of blue** — the model never sees a future month while learning to predict an earlier one. That is the time-respecting constraint that makes walk-forward CV honest. Second, **the training window grows** across folds. That growth is not a bug; it mimics real-world deployment, where you retrain monthly on an ever-growing history and always forecast into unseen time. The consequence is that earlier folds are harder and later folds are easier — which is why the per-fold MAEs in §9 will show some variation.

For the workforce planner, this means the model is tested under progressively more realistic conditions: early folds simulate forecasting with limited history; late folds simulate forecasting with decades of data — the situation the planner actually faces today.

> **A question that often comes up here:** *"Does the growing window give later folds an unfair advantage?"* Yes, and that is realistic. In deployment, you always have more history than you did six months ago. The expanding-window design captures that asymmetry honestly. A fixed-size sliding window (where you drop the oldest rows as you add new ones) is an alternative when you believe only recent history is relevant — but for a series with an 80-year trend, throwing away the early decades would discard useful signal.

With the walk-forward folds in hand, the next question is what to measure on each fold. Section 7 introduces four forecasting metrics and runs the classical benchmarks under all of them.

**The planner's conclusion:** My CV setup is honest and operationally realistic — each of the 5 expanding folds simulates a deployment scenario where I retrain monthly on accumulating history. Earlier folds are harder (less history); later folds are easier (more history) — exactly the gradient I face in production. I am ready to evaluate candidate forecasters in §7 using these folds.

---

## 7. Forecasting Metrics — What "Good" Means and What to Beat

Before evaluating any model, the workforce planner must answer two questions:

1. **Which metric defines success?** Forecast quality can be measured in several ways — and the right metric depends on how the *legislature* will use the number, not on what the textbook lists first.
2. **What baselines must the model beat?** Before fitting any learned model, the planner needs free alternatives — forecasts that require no modeling at all. If a learned model cannot beat these, there is no champion to defend.

Section 7 walks both decisions in sequence:
- **§7.1** defines the four standard forecasting metrics (MAE, RMSE, MAPE, MASE) with their formulas.
- **§7.2** gives the textbook "when to use which" guide.
- **§7.3** translates the textbook guidance into **this** business case — *which metric the planner will put on the legislative report*.
- **§7.4** explains the four classical baselines in depth (what each one assumes, when it works, when it fails, why the planner includes it).
- The code cell then runs all four baselines under all four metrics.
- **§7.5** runs the same baselines on a non-seasonal contrast (Google daily stock) to show that the right baseline depends on the data, not on the modeling assumption.

### 7.1 The four metrics

Each metric compares the actual values ($y_t$) against the forecasted values ($\hat{y}_t$) across $n$ time periods. In every formula below, $y_t$ is "what actually happened in month $t$" and $\hat{y}_t$ is "what the model predicted for month $t$."

**Mean Absolute Error (MAE).**

$$\text{MAE} = \frac{1}{n}\sum_{t=1}^{n} \left| y_t - \hat{y}_t \right|$$

The difference $y_t - \hat{y}_t$ is the forecast error for month $t$ — positive when the model under-predicted, negative when it over-predicted. The absolute value $|\ |$ strips the sign so over-predictions and under-predictions count equally. The sum adds up all $n$ absolute errors, and dividing by $n$ gives the average. The result is in the **data's original units** — thousands of employees for our series. If the MAE is 200, the model is off by about 200 thousand employees on average. Robust to occasional large errors, MAE is the workforce planner's default reporting metric.

**Root Mean Squared Error (RMSE).**

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{t=1}^{n} \left( y_t - \hat{y}_t \right)^2}$$

Same forecast error $y_t - \hat{y}_t$, but now it is **squared** before averaging. Squaring makes large errors count disproportionately more than small ones — a single month off by 500 hurts far more than five months off by 100 each, even though the total absolute error is the same. The square root at the end brings the result back to approximately the data's units. Use RMSE when a large miss is disproportionately costly — stock-outs, surge planning, safety-critical capacity.

**Mean Absolute Percentage Error (MAPE).**

$$\text{MAPE} = \frac{100}{n}\sum_{t=1}^{n} \left| \frac{y_t - \hat{y}_t}{y_t} \right|$$

The error is divided by the actual value $y_t$ before averaging, which turns it into a **percentage**. The result is scale-free and comparable across series with different magnitudes — a MAPE of 3% means "the forecast is off by 3% of the actual, on average." This is the metric that speaks to non-technical audiences ("we are off by 3%"). Two caveats: MAPE breaks if any $y_t$ is near zero (division by near-zero inflates the percentage), and it is asymmetric — over-forecasts produce larger percentages than equally-sized under-forecasts.

**Mean Absolute Scaled Error (MASE).**

$$\text{MASE} = \frac{\text{MAE}}{\text{MAE}_{\text{seasonal-naive, in-sample}}}$$

The numerator is your model's MAE; the denominator is the MAE that the seasonal-naive baseline achieves on the training data. The ratio answers a simple question: **does your model beat the free baseline?** A MASE below 1 means yes — your model's errors are smaller than what you get for free from seasonal-naive. A MASE above 1 means no — the free baseline is better than your model, and you do not have a champion. MASE is scale-free and comparable across series with different units and magnitudes.

### 7.2 When to use which

| Use this metric ... | ... when |
|---|---|
| **MAE** | Reporting in business units (employees, dollars, units sold) and you want a robust default. |
| **RMSE** | Large misses cost much more than small ones (stock-outs, surge planning, safety-critical capacity). |
| **MAPE** | Reporting to non-technical audiences ("we are off by 3% on average") — and only when *y* is far from zero across the validation window. |
| **MASE** | Comparing forecasts across multiple series with different scales (cross-region demand, multi-product KPIs). |

For the workforce planner, **MAE is the primary metric** — the legislature reads "off by 200 thousand employees on average" more easily than a percentage or a ratio. **MAPE** is the backup for the press briefing ("off by 1.5%"). **RMSE** matters if the planner's procurement contracts penalize large misses disproportionately.

**A question that often comes up here:** *"If the metrics rank models differently, which do I trust?"* The one whose error structure matches your business cost. If a stock-out costs 10× as much as overstock, RMSE is the honest metric — squaring penalizes the rare large miss exactly the way the cost matrix does. There is no "best" metric in the abstract; there is only the metric that aligns with consequences.


### 7.3 The Planner's Choice — MAE for This Business Case

Four metrics in the toolkit, but the workforce planner must pick **one** as the primary number on the legislative report. The right choice for forecasting US retail-sector employment for the legislature:

- **Primary metric: MAE.** The legislature reads *"the forecast is off by 200 thousand employees on average"* more naturally than any percentage or ratio. MAE is in the data's native units (thousands of employees) — the same units the planner uses in every other budget document. A single number in business units is harder to misinterpret than a derived statistic.
- **Secondary metric: MAPE.** For the press briefing or a non-technical committee, *"off by 1.5%"* is easier to parse than *"off by 200 thousand out of 16 million."* MAPE is the press-friendly backup. Acceptable because employment is far from zero across the validation window (recall the MAPE caveat from §7.1).
- **Diagnostic metric: RMSE.** RMSE penalizes large misses more than small ones. The planner uses it as a *check*: if MAE and RMSE rank candidate models differently, that disagreement signals heavy-tailed errors (a few months are very wrong) and the planner should investigate. **RMSE does not go on the headline slide** because retail workforce procurement contracts do not have asymmetric large-miss penalties — small misses are not disproportionately cheaper than large ones.
- **Reference metric: MASE.** MASE is a *gate*, not a headline. The planner reports it once in the methods section: *"MASE = X.XX, i.e., the model beats Seasonal-Naive by X%."* If MASE ≥ 1, there is no champion to defend — ship Seasonal-Naive and tell the legislature *"last year's pattern is the best forecast."*

**Selection table for the legislative report:**

| Metric | Used for | Where it appears |
|---|---|---|
| **MAE** | The headline forecast accuracy number | Primary slide, summary report |
| **MAPE** | Press-friendly accuracy ("off by X%") | Press release, executive summary |
| **RMSE** | Internal diagnostic (catches heavy-tailed errors) | Technical appendix only |
| **MASE** | Sanity gate (model > free baseline?) | Methods section, footnote |

This choice threads through the rest of the notebook. §8's CV comparison uses **MAE as the selection metric**; the multi-metric sensitivity table checks whether the ranking would flip under RMSE / MAPE / MASE. §9's locked-test ceremony reports MAE first, with the other three as supporting diagnostics in the legislative summary.

### 7.4 The Four Classical Baselines — What Each One Does

Each baseline encodes one specific *assumption* about what next month's employment will look like. The planner runs all four, then names which one the learned model in §8 must beat.

#### Mean

$$\hat{y}_{t+h} = \bar{y} \quad \text{for every horizon } h$$

- **What it assumes:** *"The series has no trend, no seasonality, no autocorrelation — every value is just a random draw around a fixed average."*
- **When it works:** Stationary white-noise series with no structure.
- **When it fails:** Any series with trend or seasonality (which is most business series, including this one).
- **Why the planner includes it:** As a **sanity floor**. If a learned model cannot beat the historical average, something is badly broken — the planner does not have a model, she has a constant. Mean is the lowest bar; everything beats it.

#### Naive

$$\hat{y}_{t+h} = y_t \quad \text{(carry the last observed value forward)}$$

- **What it assumes:** *"Tomorrow looks like today. The most recent observation is the best guess for the next one."*
- **When it works:** Series with strong lag-1 autocorrelation and no seasonality — random walks, daily stock prices, FX rates.
- **When it fails:** Series with strong seasonality (December's prediction would equal November's, missing the holiday peak entirely).
- **Why the planner includes it:** As a **single-feature reference**. The ACF in §3.5 confirmed strong lag-1 autocorrelation, so Naive will be a real competitor for short horizons.

#### Seasonal-Naive

$$\hat{y}_{t+h} = y_{t+h-12} \quad \text{(use the value from the same month one year ago)}$$

- **What it assumes:** *"Next December = last December. The calendar pattern repeats."*
- **When it works:** Series with strong stable seasonality and weak trend within a season.
- **When it fails:** Series with rapid trend changes or no seasonal structure (e.g., daily stock prices — §7.5).
- **Why the planner includes it:** As the **primary competitor**. The ACF showed lag-12 is a dominant signal. Seasonal-Naive is the bar the learned model in §8 must beat with non-overlapping CIs, or the planner ships Seasonal-Naive itself.

#### Drift

$$\hat{y}_{t+h} = y_t + h \cdot \frac{y_t - y_1}{t - 1}$$

(straight line from the first observation to the last, extended forward)

- **What it assumes:** *"The series grows at a constant rate. Tomorrow continues the average trend."*
- **When it works:** Series with a steady linear trend and no seasonality.
- **When it fails:** Series with seasonality (Drift captures the upward direction but misses the annual cycle entirely).
- **Why the planner includes it:** As a **trend reference**. Retail employment has a strong long-run trend; Drift captures the trend direction. If a learned model cannot beat Drift, the linear trend is the entire forecastable signal.

**Implementation note.** All four baselines are one-liners using `sktime`'s `NaiveForecaster` with the appropriate `strategy` parameter (`"mean"`, `"last"`, `"last"` with `sp=12`, and `"drift"`). The code cell below fits each one on the demo training window and forecasts the held-out validation portion under all four metrics — but remember, **MAE is the metric the planner will defend**.

> 💡 **Gemini Prompt:** *"Build a dict `baselines` with four sktime forecasters: `"Mean"`: `NaiveForecaster(strategy="mean")`, `"Naive"`: `NaiveForecaster(strategy="last")`, `"Seasonal-Naive"`: `NaiveForecaster(strategy="last", sp=12)`, `"Drift"`: `NaiveForecaster(strategy="drift")`. Split `y_train` into a demonstration fit window (`y_demo_fit` = first 75%) and a held-out evaluation window (`y_demo_eval` = last 25%). Set `fh_demo = np.arange(1, len(y_demo_eval) + 1)`. Fit each baseline on `y_demo_fit` and forecast `fh_demo`, storing predictions in a dict `preds`. Plot the last 60 months of `y_demo_fit` in black, the actuals in `y_demo_eval` as a black dashed line, and overlay each baseline's prediction line on a `figsize=(12, 5)` figure with legend in upper left. Then define an `all_metrics(y_true, y_pred, training_y, season=12)` function returning MAE, RMSE, MAPE, and MASE. Build a DataFrame `benchmark_table` with one row per baseline and one column per metric; print it rounded to 3 decimals and the per-metric ranking."*
>
> **After running, verify:**
> - [ ] Four colored lines crossing the validation window
> - [ ] Mean is a flat horizontal line; Seasonal-Naive replays a wave
> - [ ] The benchmark_table has 4 rows (one per baseline) and 4 columns
> - [ ] Seasonal-Naive ranks first or near-first on MAE for retail data

In [ ]:
# Four classical benchmarks using sktime's NaiveForecaster.
# Each strategy maps to the manual function we would have written by hand.
baselines = {
    "Mean":          NaiveForecaster(strategy="mean"),
    "Naive":         NaiveForecaster(strategy="last"),
    "Seasonal-Naive": NaiveForecaster(strategy="last", sp=12),
    "Drift":         NaiveForecaster(strategy="drift"),
}

# Fit each baseline on the training data and forecast the validation window
# For the visual demo, fit on the first 75% of the training window
# and forecast the last 25% — a single-shot preview before CV.
n_demo = int(len(y_train) * 0.75)
y_demo_fit = y_train.iloc[:n_demo]
y_demo_eval = y_train.iloc[n_demo:]
fh_demo = np.arange(1, len(y_demo_eval) + 1)
preds = {}
for name, model in baselines.items():
    model.fit(y_demo_fit)
    preds[name] = model.predict(fh=fh_demo).values

# Plot all four against the validation actuals
fig, ax = plt.subplots(figsize=(12, 5))
train_tail = y_demo_fit.iloc[-60:]
ax.plot(train_tail.index.to_timestamp(), train_tail.values, color="black", label="Fit window (last 60 mo)")
ax.plot(y_demo_eval.index.to_timestamp(), y_demo_eval.values, color="black", linestyle="--", label="Held-out (actual)")
for name, p in preds.items():
    ax.plot(y_demo_eval.index.to_timestamp(), p, label=name)
ax.set_title("Four classical benchmarks on the validation window")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

# --- Multi-metric evaluation utility ---
def all_metrics(y_true, y_pred, training_y, season=12):
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    mae = np.mean(np.abs(yt - yp))
    rmse = np.sqrt(np.mean((yt - yp) ** 2))
    mape = np.mean(np.abs((yt - yp) / yt)) * 100.0
    th = np.asarray(training_y, dtype=float)
    seasonal_naive_errors = np.abs(th[season:] - th[:-season])
    mae_naive = seasonal_naive_errors.mean()
    mase = mae / mae_naive if mae_naive > 0 else np.nan
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "MASE": mase}

benchmark_table = pd.DataFrame({
    name: all_metrics(y_demo_eval.values, p, y_demo_fit.values)
    for name, p in preds.items()
}).T
print("Four classical benchmarks \u2014 single-shot validation, all four metrics:")
print(benchmark_table.round(3))

print("\nRanking under each metric (1 = best):")
print(benchmark_table.rank(axis=0).astype(int))

**Reading the output:**

Three outputs to read in sequence: the plot, the metric table, and the ranking table.

**The plot** shows five lines crossing the validation window: the four baseline forecasts and the black dashed actual values. Look for which colored line tracks the black actuals most closely. On US Retail Trade, **Seasonal-Naive** (which replays the last 12 months of training forward) typically hugs the actuals best — it captures the annual peaks and troughs that the other baselines miss. **Mean** is visibly the worst — it draws a flat horizontal line through the middle of the validation window, missing both the trend and the seasonality. **Naive** (carry the last training value forward) captures the level but misses the seasonal swing. **Drift** (straight-line extrapolation) captures the trend direction but also misses the seasonality.

**The metric table** puts numbers behind the visual impression. Four rows (one per baseline), four columns (MAE, RMSE, MAPE, MASE). Read the MAE column first — it is in the planner's native units (thousands of employees). Seasonal-Naive typically has the lowest MAE, confirming what the plot showed. The MASE column is the reality check: a MASE below 1.0 means the baseline beats the in-sample seasonal-naive benchmark; above 1.0 means it lost. By definition, Seasonal-Naive's MASE is near 1.0 (it *is* the reference baseline for that metric).

**The ranking table** shows where the four metrics agree and where they disagree. When all four metrics rank the same baseline first, the ranking is **robust** — you can state the winner with confidence. When rankings disagree (say, MAE picks Seasonal-Naive but RMSE picks Drift), the disagreement is the teaching moment: MAE treats every error equally, while RMSE punishes the occasional large miss more heavily. The choice between them is a **business** choice — does the workforce planner's procurement plan tolerate steady small misses (favor MAE) or is a single large miss catastrophic (favor RMSE)?

These rankings depend on the data's structural features. The next subsection makes that concrete by running the same benchmarks on a series with no seasonality at all.

**The planner's conclusion:** Seasonal-Naive — "December 2018 will look like December 2017" — is the bar my learned model must beat in §8. This is the free alternative the legislature will rightly ask about, and any model I propose has to convincingly outperform it on the CI overlap rule. If my Linear [lag1, lag12] cannot beat Seasonal-Naive's CI in §8, I will ship Seasonal-Naive and tell the legislature: *"Last year's pattern is the best defensible forecast."* Before §8, the contrast in §7.5 reminds me that the right baseline depends on the data.

---

### 7.5 Non-Seasonal Contrast — Google Daily Stock Prices

The US Retail Trade series above has clear annual seasonality, which is why **Seasonal-Naive** was such a strong baseline. But many business series — daily stock prices, intraday traffic, hourly server load, web-conversion rates — have little or no calendar seasonality. The classical benchmarks behave very differently there: **Drift** (linear extrapolation from start to end of training) often beats Seasonal-Naive by a lot because there is no annual pattern to lean on, and **Naive** (carry the last training value forward) can also be competitive because consecutive days are highly correlated.

To make the contrast concrete, we run the same benchmarks on Google daily closing prices: train on 2015, test on January 2016. The pedagogical point is that **the choice of benchmark depends on the data’s structural features, not on the model**.

> 💡 **Gemini Prompt:** *"Read the GAFA stock CSV from `https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/lecture_slides/08_time_series/data/gafa_stock.csv` with `parse_dates=["ds"]`. Filter to `gafa["unique_id"] == "GOOG_Close"`, keep `ds` and `y` columns, sort, reset_index. Set `ds` as the index of a Series `y_goog`. Slice `goog_train = y_goog["2015"]` and `goog_test = y_goog["2016-01"]`, then convert each index to `PeriodIndex("D")` for sktime compatibility. Print train/test date ranges. Build three baselines (`Mean`, `Naive` = strategy `last`, `Drift`) using `NaiveForecaster` on `goog_train`, forecasting `fh_g = np.arange(1, len(goog_test) + 1)`. Plot 2015 train in grey, January 2016 actuals in black, and each baseline prediction in color, with a red dotted vertical line at the 2016-01-01 boundary. Use `all_metrics(..., season=1)` for MASE referenced to a 1-step naive baseline. Print the table and the per-metric ranking."*
>
> **After running, verify:**
> - [ ] Train n is roughly 252 trading days; test n is roughly 19
> - [ ] Drift typically ranks first; Mean ranks last
> - [ ] Seasonal-Naive is absent (no annual cycle for daily stock)
> - [ ] Ranking flips relative to the US Retail Trade result

In [ ]:
# Load Google daily closing prices from the GAFA stock dataset
GAFA_URL = (
    "https://raw.githubusercontent.com/davi-moreira/"
    "2026Summer_predictive_analytics_purdue_MGMT474/main/"
    "lecture_slides/08_time_series/data/gafa_stock.csv"
)
gafa = pd.read_csv(GAFA_URL, parse_dates=["ds"])
goog = (
    gafa[gafa["unique_id"] == "GOOG_Close"]
    .loc[:, ["ds", "y"]]
    .sort_values("ds")
    .reset_index(drop=True)
)

# Convert to sktime format. Stock data trades on irregular days
# (weekends and holidays are missing). PeriodIndex("D") gives each
# observation a daily period label without requiring a regular grid,
# so sktime can compute forecast horizons.
y_goog = goog.set_index("ds")["y"]

# Train: 2015 (full calendar year). Test: January 2016.
goog_train = y_goog["2015"]
goog_test  = y_goog["2016-01"]
goog_train.index = goog_train.index.to_period("D")
goog_test.index  = goog_test.index.to_period("D")

print(f"Train: {goog_train.index[0]} -> {goog_train.index[-1]}  (n={len(goog_train)})")
print(f"Test : {goog_test.index[0]} -> {goog_test.index[-1]}  (n={len(goog_test)})")

# Three classical benchmarks using NaiveForecaster
fh_g = np.arange(1, len(goog_test) + 1)
goog_baselines = {
    "Mean":  NaiveForecaster(strategy="mean"),
    "Naive": NaiveForecaster(strategy="last"),
    "Drift": NaiveForecaster(strategy="drift"),
}
preds_g = {}
for name, model in goog_baselines.items():
    model.fit(goog_train)
    preds_g[name] = model.predict(fh=fh_g).values

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(goog_train.index.to_timestamp(), goog_train.values,
        color="grey", alpha=0.6, linewidth=0.8, label="2015 (train)")
ax.plot(goog_test.index.to_timestamp(), goog_test.values,
        color="black", linewidth=1.5, label="Jan 2016 (test, actual)")
for name, p in preds_g.items():
    ax.plot(goog_test.index.to_timestamp(), p, label=name, linewidth=1.2)
ax.axvline(pd.Timestamp("2016-01-01"), color="red", linestyle=":", alpha=0.5,
           label="Train / test boundary")
ax.set_title("Google Daily Closing Price \u2014 2015 Train, Jan 2016 Test")
ax.set_xlabel("Date")
ax.set_ylabel("Closing price (USD)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

goog_table = pd.DataFrame({
    name: all_metrics(goog_test.values, p, goog_train.values, season=1)
    for name, p in preds_g.items()
}).T
print("\nAccuracy table \u2014 Google daily, January 2016 horizon (MASE referenced to 1-step naive):")
print(goog_table.round(3))
print("\nRanking under each metric (1 = best):")
print(goog_table.rank(axis=0).astype(int))

**Reading the output:**

The ranking on Google daily prices typically **flips** relative to US Retail Trade — a concrete demonstration that the right baseline depends on the data, not on a modeling assumption.

**In the plot**, the 2015 training data appears in grey and the January 2016 test window in black. The three forecast lines extend past the red vertical cutoff. **Drift** (the straight-line extrapolation from the first to the last training price) typically tracks the January actuals most closely — it captures the mild upward or downward momentum of the 2015 price trajectory. **Naive** (carry the last December 2015 closing price forward) draws a flat line that is competitive because consecutive trading days are highly correlated; tomorrow's price is almost always close to today's. **Mean** (the average of all 2015 closing prices) is usually the worst — it projects a price from mid-2015 into January 2016, throwing away the level the stock actually closed at.

**Why the flip?** US Retail Trade has strong annual seasonality, so Seasonal-Naive — which replays last year's pattern — is the natural winner. Google daily stock has no calendar seasonality (there is no "December peak" in closing prices), so Seasonal-Naive is not even in the race. The dominant structure is a slow random-walk-like drift, which is exactly what the Drift baseline captures.

**MASE with season = 1** deserves a note. For the retail employment series, MASE used the 12-step seasonal-naive baseline as the denominator. For a non-seasonal series, the natural denominator is the 1-step naive baseline (a random walk). The formula is the same; only the reference baseline changes. A MASE below 1 still means "the model beats the free baseline" — the baseline is just a different one.

The business takeaway: **build the classical benchmarks first on every new series; let the data tell you which one your learned model has to beat.** A model evaluated against the wrong baseline can look impressive while adding no real value.

With the benchmarks established on both seasonal and non-seasonal series, section 8 asks: can a simple learned model — linear regression on lag features — beat those free baselines?

**The planner's conclusion:** The Google ranking flip reinforces my methodology: I do not choose a baseline a priori, I let the data tell me which one to beat. For US Retail Trade, Seasonal-Naive is the bar; for a non-seasonal series, it would be Drift. If the legislature later asks me to forecast a different series (intermittent demand, daily indicators), I will run §7's baseline diagnostic before picking a champion. Now §8 runs the head-to-head: can my Linear [lag1, lag12] model beat the free Seasonal-Naive baseline?

---

## 8. Cross-Validated Comparison — Five Candidates, Identical Folds

The baselines in §7 gave us a visual preview, but a single-split evaluation is one roll of the dice. We now compare all five forecasters on the **same** walk-forward folds from §6.

Three of the five candidates are the `NaiveForecaster` baselines from §7. The other two are **learned models** that use the ACF-driven feature selection from §3.5: a `LinearRegression` and a `Ridge`, each fitted on exactly **lag-1 and lag-12** — the two lags the ACF identified as carrying the strongest signal.

To build these models, we combine two tools from earlier in the course. First, `make_reduction` from `sktime` constructs a 12-column lag matrix inside its `.fit()` call (one column per lag, from `lag1` through `lag12`). Second, a sklearn `Pipeline` — the same pattern nb02 taught — selects only columns 0 and 11 (lag-1 and lag-12) before feeding them to the regressor. The result is a forecaster that:

- uses exactly the two features the ACF justified,
- constructs them inside `.fit()` so they respect fold boundaries during CV (no leakage by construction), and
- handles recursive multi-step forecasting automatically (`strategy="recursive"` feeds predictions back as lag inputs).

This is the **Pipeline principle from nb02** applied to forecasting: feature engineering inside the model, not outside it. For the workforce planner, this means the forecast is **reproducible and auditable** — anyone can re-run the CV loop and verify the results, because the entire feature-engineering and model-fitting process is encapsulated in a single forecaster object.

> 💡 **Gemini Prompt:** *"Define `select_lag1_lag12(X)` that returns `X[:, [0, 11]]` (lag-1 is column 0, lag-12 is column 11 in make_reduction's 12-column matrix). Build two sklearn `Pipeline` objects: `lr_pipeline` with steps `("select_lags", FunctionTransformer(select_lag1_lag12))` and `("regressor", LinearRegression())`; and `ridge_pipeline` with the same selector and `Ridge(alpha=1.0, random_state=RANDOM_SEED)`. Build a dict `candidates` with five forecasters: `Mean`, `Naive`, `Seasonal-Naive` (NaiveForecasters), `Linear [lag1,lag12]` and `Ridge  [lag1,lag12]` (`make_reduction(<pipeline>, window_length=12, strategy="recursive")`). Run walk-forward CV: for each candidate, clone before each fold, fit on `y_train.iloc[tr_idx]`, predict `fh=np.arange(1, len(va_idx)+1)`, store fold MAEs in a DataFrame `results`. Compute the Student's `t` 95% CI half-width with `t_crit = student_t.ppf(0.975, df=n_folds-1)` and build a `summary` DataFrame sorted by `MAE_mean`. Also run a per-fold multi-metric check (MAE, RMSE, MAPE, MASE), reorder to match `summary.index`, and print the ranking. Plot a horizontal bar chart with error bars equal to the CI half-widths. Finally refit `lr_pipeline` in a make_reduction champion and print its 2 coefficients (lag1, lag12) and intercept."*
>
> **After running, verify:**
> - [ ] `summary` has 5 rows sorted by MAE_mean (lowest first)
> - [ ] Linear [lag1,lag12] and Ridge [lag1,lag12] are near the top
> - [ ] Mean has by far the worst MAE (sanity floor)
> - [ ] The 2 printed coefficients are roughly lag1 \u2248 0.5, lag12 \u2248 0.5

In [ ]:
# ACF-driven lag selection: pick only lag-1 and lag-12 from the
# 12-column lag matrix that make_reduction constructs.
def select_lag1_lag12(X):
    """Pick lag-1 (column 0) and lag-12 (column 11)."""
    return X[:, [0, 11]]

# Build sklearn Pipelines (same pattern as nb02):
# Step 1: select the two ACF-justified lags
# Step 2: fit the regressor on those two features only
lr_pipeline = Pipeline([
    ("select_lags", FunctionTransformer(select_lag1_lag12)),
    ("regressor", LinearRegression()),
])
ridge_pipeline = Pipeline([
    ("select_lags", FunctionTransformer(select_lag1_lag12)),
    ("regressor", Ridge(alpha=1.0, random_state=RANDOM_SEED)),
])

# Five-candidate walk-forward comparison on identical folds.
candidates = {
    "Mean":               NaiveForecaster(strategy="mean"),
    "Naive":              NaiveForecaster(strategy="last"),
    "Seasonal-Naive":     NaiveForecaster(strategy="last", sp=12),
    "Linear [lag1,lag12]": make_reduction(lr_pipeline, window_length=12,
                                          strategy="recursive"),
    "Ridge  [lag1,lag12]": make_reduction(ridge_pipeline, window_length=12,
                                          strategy="recursive"),
}

# --- Walk-forward CV loop ---
results = pd.DataFrame()
for name, forecaster in candidates.items():
    fold_maes = []
    for tr_idx, va_idx in cv.split(y_train):
        y_cv_train = y_train.iloc[tr_idx]
        y_cv_val   = y_train.iloc[va_idx]
        fc = forecaster.clone()
        fc.fit(y_cv_train)
        y_cv_pred = fc.predict(fh=np.arange(1, len(y_cv_val) + 1))
        fold_maes.append(mean_absolute_error(y_cv_val, y_cv_pred))
    results[name] = np.array(fold_maes)

# --- Selection metric (MAE) with Student's t 95% CI ---
t_crit = student_t.ppf(0.975, df=n_folds - 1)
summary = pd.DataFrame({
    "MAE_mean": results.mean(),
    "MAE_sd":   results.std(ddof=1),
    "CI_halfwidth": results.std(ddof=1) / np.sqrt(n_folds) * t_crit,
}).sort_values("MAE_mean")
summary["CI_low"] = summary["MAE_mean"] - summary["CI_halfwidth"]
summary["CI_high"] = summary["MAE_mean"] + summary["CI_halfwidth"]
print("Selection metric (MAE) \u2014 walk-forward CV with 95% CI:")
print(summary.round(2))

# --- Multi-metric sensitivity check ---
def per_fold_all_metrics(forecaster, y_data, cv_splitter, season=12):
    out = {"MAE": [], "RMSE": [], "MAPE": [], "MASE": []}
    for tr_idx, va_idx in cv_splitter.split(y_data):
        y_tr = y_data.iloc[tr_idx]
        y_va = y_data.iloc[va_idx]
        fc = forecaster.clone()
        fc.fit(y_tr)
        yp = fc.predict(fh=np.arange(1, len(y_va) + 1)).values
        yt = y_va.values
        out["MAE"].append(np.mean(np.abs(yt - yp)))
        out["RMSE"].append(np.sqrt(np.mean((yt - yp) ** 2)))
        out["MAPE"].append(np.mean(np.abs((yt - yp) / yt)) * 100.0)
        th = y_tr.values
        if len(th) > season:
            mae_naive = np.mean(np.abs(th[season:] - th[:-season]))
            out["MASE"].append(np.mean(np.abs(yt - yp)) / mae_naive)
        else:
            out["MASE"].append(np.nan)
    return {k: np.array(v) for k, v in out.items()}

all_results = {name: per_fold_all_metrics(fc, y_train, cv)
               for name, fc in candidates.items()}
metric_means = pd.DataFrame({m: {name: r[m].mean() for name, r in all_results.items()}
                             for m in ["MAE", "RMSE", "MAPE", "MASE"]})
metric_means = metric_means.loc[summary.index]
print("\nMulti-metric sensitivity check \u2014 per-candidate mean:")
print(metric_means.round(3))
print("\nRanking under each metric (1 = best):")
print(metric_means.rank(axis=0).astype(int))

# --- Selection bar chart ---
fig, ax = plt.subplots(figsize=(11, 5))
y_pos = np.arange(len(summary))
ax.barh(y_pos, summary["MAE_mean"],
        xerr=summary["CI_halfwidth"], color="#1f77b4", edgecolor="black", capsize=4)
ax.set_yticks(y_pos)
ax.set_yticklabels(summary.index)
ax.invert_yaxis()
ax.set_xlabel("MAE (walk-forward CV; bars = 95% CI)")
ax.set_title("Five-candidate forecast comparison \u2014 selection metric (MAE)")
plt.tight_layout()
plt.show()

# --- Champion's coefficients (2 values: lag1 and lag12) ---
champ_fc = make_reduction(lr_pipeline, window_length=12, strategy="recursive")
champ_fc.fit(y_train)
coefs = champ_fc.estimator_.named_steps["regressor"].coef_
intercept = champ_fc.estimator_.named_steps["regressor"].intercept_
print(f"\nChampion coefficients:  lag1 = {coefs[0]:.3f},  lag12 = {coefs[1]:.3f}")
print(f"Intercept: {intercept:.1f}")
print("The ACF predicted lag-1 and lag-12 would dominate — these two")
print("coefficients ARE the entire model.")

**Reading the output:**

Three outputs, read in sequence.

**The MAE selection table** is the primary decision tool. Each row is one candidate model; the columns show the mean MAE across five walk-forward folds, the standard deviation, the CI half-width, and the lower and upper bounds of the 95% CI. The candidates are sorted by mean MAE — the top row is the current leader. Look at the CI columns: if the leader's CI does not overlap with the runner-up's CI, the leader is **genuinely better** on this metric. If the CIs overlap, you cannot distinguish them statistically — pick the simpler model.

**The multi-metric sensitivity table** shows the mean score for each candidate under all four metrics (MAE, RMSE, MAPE, MASE). The ranking table below it asks *"would I pick a different champion if I cared about a different metric?"* If all four metrics rank the same model first, the champion is **robust** — ship it with confidence. If the rankings disagree (model A wins on MAE but model B wins on RMSE), the disagreement points you to the metric whose error structure matches the business cost.

**The bar chart** makes the CI comparison visual. Horizontal bars show each candidate's mean MAE; error bars show the 95% CI. Look for gaps between bars: a clear gap with no error-bar overlap means a genuine difference; overlapping error bars mean statistically indistinguishable candidates.



**The champion's coefficients** confirm the ACF analysis: the model has exactly two weights — one for lag-1 (short-term momentum) and one for lag-12 (annual seasonality). Together with the intercept, these three numbers ARE the entire model. The workforce planner can read them directly: "each additional thousand employees last month contributes about X hundred to next month's forecast; each additional thousand from the same month last year contributes about Y hundred." No black box — any analyst can verify the forecast by hand.

Three interpretation rules borrowed from nb08:

1. **Non-overlapping CIs** between candidate A and candidate B → A is genuinely better on the selection metric.
2. **Overlapping CIs** → no statistical evidence to prefer one over the other; pick the simpler model (Occam's razor).
3. **Mean is far worse than the rest** → expected. It ignores trend and seasonality entirely; it is only here as a sanity floor.

**The planner's conclusion:** the champion is the model with the lowest mean MAE and non-overlapping CIs against its nearest competitor. This is the model she will refit on the full training window and present to the legislature in §9's ceremony.

If the linear and Ridge models have overlapping CIs, **Ridge does not earn its place** here — the regularization adds machinery without a measurable payoff. That is the right outcome for a 2-feature model; Ridge typically wins when the feature count is large and multicollinearity is a real risk.

---

## 📝 PAUSE-AND-DO Exercise 1 — Compare [lag1, lag12] vs all 12 lags (10 minutes)

**Task:** The ACF-driven model uses only lag-1 and lag-12. But `make_reduction(LinearRegression(), window_length=12)` without the lag-selection Pipeline uses *all 12 lags*. Does the full 12-lag model earn its place over the 2-lag model by **non-overlapping CIs**?

**Hints:**
- Build `make_reduction(LinearRegression(), window_length=12, strategy="recursive")` — no Pipeline, no lag selection.
- Run it through the same CV folds and add its fold MAEs to the results DataFrame as `"Linear [all 12 lags]"`.
- Rebuild the summary table with all six candidates and compare CIs.
- Overlap = the extra lags did not earn their place; the ACF-driven 2-feature model is sufficient.

Type your code in the cell below.

> 💡 **Gemini Prompt:** *"I have a walk-forward CV comparison of five forecasters on monthly US retail employment data using sktime's ExpandingWindowSplitter (cv). The candidates include NaiveForecaster baselines and make_reduction with a Pipeline that selects only lag-1 and lag-12 from a 12-lag matrix. I want to add a sixth candidate: make_reduction(LinearRegression(), window_length=12, strategy='recursive') — a model using ALL 12 lags without the lag-selection step. Run it through the same CV folds using .clone() / .fit() / .predict(), add its per-fold MAEs to the results DataFrame as 'Linear [all 12 lags]', rebuild the summary table with Student's t 95% CIs (t_crit and n_folds are already defined), and print the updated table plus a horizontal bar chart comparing all six candidates."*
>
> **After running, verify:**
> - [ ] The new model `Linear [all 12 lags]` appears in the summary table alongside the original five candidates
> - [ ] The CI for the 12-lag model overlaps (or does not overlap) with `Linear [lag1,lag12]` — note which
> - [ ] The bar chart shows error bars for all six candidates
> - [ ] No test-set data was used anywhere

In [ ]:
# YOUR SOLUTION CODE HERE

# Hints:
# lr_all12 = make_reduction(LinearRegression(), window_length=12, strategy="recursive")
# fold_maes_12 = []
# for tr_idx, va_idx in cv.split(y_train):
#     fc = lr_all12.clone()
#     fc.fit(y_train.iloc[tr_idx])
#     pred = fc.predict(fh=np.arange(1, len(va_idx) + 1))
#     fold_maes_12.append(mean_absolute_error(y_train.iloc[va_idx], pred))
# results_ex1 = results.copy()
# results_ex1["Linear [all 12 lags]"] = np.array(fold_maes_12)
# Build the updated summary table, compare CIs.

## 9. Opening the Locked Test Window — The Planner's One-Shot Evaluation

This is the moment the workforce planner has been building toward across the entire notebook. The EDA in §3 identified the three structural components (trend, seasonality, structural breaks). The ACF in §3.5 selected lag-1 and lag-12 as the features. The walk-forward CV in §8 compared five candidates and named the champion. Now the planner refits that champion on the full 957-month training window and predicts the **locked 12-month test window** — the most recent year the model has never seen.

The planner reads six outputs and assembles the legislative report:

1. **The verdict** — does the test MAE fall INSIDE the CV 95% CI? If yes, the selection process was honest and the planner can stand behind the model.
2. **The prediction interval** — a 95% Gaussian band around each monthly forecast, estimated from walk-forward residuals (not in-sample residuals, which would be too narrow).
3. **The full-series plot** — the 80-year context figure.
4. **The zoomed plot** — test window + 24 months of training context, with monthly markers. This is the poster figure.
5. **The horizon curve** — RMSE at each forecast step (1–12 months). The planner quotes this when the legislature asks *"how far ahead can we trust this?"*
6. **The legislative summary** — a printed report block packaging all diagnostics in the format a decision-maker reads.

The workforce planner walks into the hearing with a complete package: point forecast, uncertainty band, reliability check, horizon sensitivity, known limitations, and a retraining recommendation. That package — not a single number — is what earns credibility.

> 💡 **Gemini Prompt:** *"Run the five-step locked-test ceremony for the `Linear [lag1,lag12]` champion. Step 1: in a loop over `cv.split(y_train)`, clone `make_reduction(lr_pipeline, window_length=12, strategy="recursive")`, fit on the fold's train, predict the fold's val, and append residuals to `fold_residuals`. Compute `sigma_residual = fold_residuals.std(ddof=1)` and print it. Step 2: refit `champion = make_reduction(lr_pipeline, window_length=12, strategy="recursive")` on the full `y_train`. Step 3: predict the 12-month locked test with `fh_test = np.arange(1, len(y_test) + 1)` and compute `test_mae = mean_absolute_error(y_test, y_test_pred)`. Step 4: build a 95% Gaussian PI with `z_95 = 1.96` times `sigma_residual` above and below `y_test_pred.values`. Step 5: compute empirical `inside` coverage and print the INSIDE/ABOVE/BELOW verdict using `summary.loc["Linear [lag1,lag12]"]`. Then plot two figures: (a) the full-series view with train (`#1f77b4`), test actuals (`#d62728`), forecast dashed `#2ca02c`, and PI band shaded `#2ca02c` alpha 0.20; (b) a zoomed view of the last 24 months of train plus the test window with marker `o` for actuals, marker `s` for forecasts, and a grey dotted boundary line. Then sweep horizons `h = 1..12` recursively across CV folds, plot the RMSE-vs-horizon curve in `#9467bd` and print the per-horizon table. Finally print the WORKFORCE PLANNER'S FORECAST REPORT block with model, forecast window, test MAE, verdict, PI width, coverage, 1-month and 12-month RMSE, limitation, and the four-part retraining policy."*
>
> **After running, verify:**
> - [ ] The verdict prints INSIDE, ABOVE, or BELOW the CV 95% CI
> - [ ] Empirical PI coverage is reported as a percentage
> - [ ] Both full-series and zoomed plots render with PI band
> - [ ] Horizon curve rises monotonically from h=1 to h=12
> - [ ] Legislative summary report block prints in a boxed format

In [ ]:
# Step 1: collect walk-forward residuals for PI sigma estimation.
fold_residuals = []
for tr_idx, va_idx in cv.split(y_train):
    y_cv_train = y_train.iloc[tr_idx]
    y_cv_val   = y_train.iloc[va_idx]
    fc = make_reduction(lr_pipeline, window_length=12, strategy="recursive")
    fc.fit(y_cv_train)
    pred_fold = fc.predict(fh=np.arange(1, len(y_cv_val) + 1))
    fold_residuals.extend(y_cv_val.values - pred_fold.values)
fold_residuals = np.array(fold_residuals)
sigma_residual = fold_residuals.std(ddof=1)
print(f"Walk-forward residual sigma: {sigma_residual:.2f} (units: thousands of employees)")

# Step 2: refit champion on the full training window
champion = make_reduction(lr_pipeline, window_length=12, strategy="recursive")
champion.fit(y_train)

# Step 3: point forecast on the locked test window
fh_test = np.arange(1, len(y_test) + 1)
y_test_pred = champion.predict(fh=fh_test)
test_mae = mean_absolute_error(y_test, y_test_pred)

# Step 4: 95% prediction interval (Gaussian assumption on walk-forward residuals)
z_95 = 1.96
y_test_lower = y_test_pred.values - z_95 * sigma_residual
y_test_upper = y_test_pred.values + z_95 * sigma_residual

# Step 5: empirical coverage
inside = ((y_test.values >= y_test_lower) & (y_test.values <= y_test_upper)).mean()

# Verdict
champ_row = summary.loc["Linear [lag1,lag12]"]
cv_low, cv_high = champ_row["CI_low"], champ_row["CI_high"]
verdict = ("INSIDE the CV 95% CI" if cv_low <= test_mae <= cv_high
           else "ABOVE the CV 95% CI (overfitting?)" if test_mae > cv_high
           else "BELOW the CV 95% CI (lucky test window?)")

print(f"\nChampion: Linear [lag1,lag12]")
print(f"CV MAE 95% CI : [{cv_low:.2f}, {cv_high:.2f}]")
print(f"Test MAE      : {test_mae:.2f}  ->  {verdict}")
print(f"Empirical 95% PI coverage on test: {inside*100:.1f}%  (nominal: 95.0%)")

# Plot
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(y_train.index.to_timestamp(), y_train.values, color="#1f77b4",
        label="Train", linewidth=0.8)
ax.plot(y_test.index.to_timestamp(), y_test.values, color="#d62728",
        label="Test (actual)", linewidth=1.5)
ax.plot(y_test.index.to_timestamp(), y_test_pred.values, color="#2ca02c",
        linestyle="--", label="Champion forecast", linewidth=1.5)
ax.fill_between(y_test.index.to_timestamp(), y_test_lower, y_test_upper,
                color="#2ca02c", alpha=0.20, label="95% prediction interval")
ax.set_title("Locked Test Window \u2014 Forecast + 95% Prediction Interval")
ax.legend()
plt.tight_layout()
plt.show()

# Zoomed view: test window + 24 months of training context
train_tail_24 = y_train.iloc[-24:]
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(train_tail_24.index.to_timestamp(), train_tail_24.values,
        color="#1f77b4", marker="o", markersize=4, label="Train (last 24 mo)")
ax.plot(y_test.index.to_timestamp(), y_test.values, color="#d62728",
        marker="o", markersize=5, label="Test (actual)", linewidth=1.5)
ax.plot(y_test.index.to_timestamp(), y_test_pred.values, color="#2ca02c",
        marker="s", markersize=5, linestyle="--",
        label="Champion forecast", linewidth=1.5)
ax.fill_between(y_test.index.to_timestamp(), y_test_lower, y_test_upper,
                color="#2ca02c", alpha=0.20, label="95% prediction interval")
ax.axvline(y_train.index[-1].to_timestamp(), color="grey", linestyle=":",
           alpha=0.7, label="Train / test boundary")
ax.set_title("Zoomed: Test Window + 24 Months of Context")
ax.set_xlabel("Month")
ax.set_ylabel("Employment (thousands)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

# --- Horizon diagnostic: how does accuracy degrade with distance? ---
HORIZONS = list(range(1, 13))
errors_by_h = {h: [] for h in HORIZONS}

for tr_idx, va_idx in cv.split(y_train):
    y_cv_train = y_train.iloc[tr_idx]
    y_cv_val   = y_train.iloc[va_idx]
    fc = make_reduction(lr_pipeline, window_length=12, strategy="recursive")
    fc.fit(y_cv_train)
    h_max = min(12, len(y_cv_val))
    preds_h = fc.predict(fh=np.arange(1, h_max + 1))
    actuals_h = y_cv_val.iloc[:h_max]
    for h in range(1, h_max + 1):
        errors_by_h[h].append((actuals_h.iloc[h-1] - preds_h.iloc[h-1]) ** 2)

rmse_by_h = {h: float(np.sqrt(np.mean(es))) for h, es in errors_by_h.items() if es}

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(rmse_by_h.keys()), list(rmse_by_h.values()),
        "o-", color="#9467bd", linewidth=2, markersize=8)
ax.set_xlabel("Forecast horizon h (months ahead)")
ax.set_ylabel("RMSE (recursive forecast)")
ax.set_title("Forecast accuracy degrades with horizon \u2014 Linear [lag1,lag12]")
ax.grid(alpha=0.3)
ax.set_xticks(list(rmse_by_h.keys()))
plt.tight_layout()
plt.show()

rmse_table = pd.Series(rmse_by_h, name="RMSE").round(2).to_frame()
rmse_table.index.name = "h (months ahead)"
print(rmse_table)

# === WORKFORCE PLANNER'S LEGISLATIVE SUMMARY ===
print("\n" + "=" * 65)
print("WORKFORCE PLANNER\u2019S FORECAST REPORT FOR THE LEGISLATURE")
print("=" * 65)
print(f"Model           : Linear regression on lag-1 and lag-12")
print(f"Forecast window : {y_test.index[0]} through {y_test.index[-1]} (12 months)")
print(f"Test MAE        : {test_mae:.0f} thousand employees")
print(f"Test verdict    : {verdict}")
print(f"95% PI width    : \u00b1{z_95 * sigma_residual:.0f} thousand employees")
print(f"PI reliability  : {inside*100:.0f}% of test actuals inside the band (nominal 95%)")
print(f"1-month RMSE    : {rmse_by_h[1]:.0f} thousand employees")
print(f"12-month RMSE   : {rmse_by_h[12]:.0f} thousand employees")
print(f"Limitation      : Cannot anticipate recession-driven shocks")
print("-" * 65)
print("Retraining policy:")
print("  Default cadence    : Monthly (fresh BLS release each month)")
print("  Practical fallback : Quarterly (matches legislative cycle)")
print("  Drift triggers     : Retrain immediately if any of:")
print("    \u00b7 Running MAE exceeds CV 95% CI upper bound")
print("    \u00b7 Recession or policy shock occurs")
print("    \u00b7 Empirical PI coverage drops below 90% over 12 months")
print("  Annual review      : Re-run ACF on the most recent decade;")
print("                       revisit feature selection if lag-1 and")
print("                       lag-12 are no longer dominant.")
print("=" * 65)

**Reading the output:**

This section produces six outputs. Read them as the workforce planner building her legislative report — each output answers a specific question the legislature will ask.

**1. The verdict (INSIDE / ABOVE / BELOW).** The test MAE is compared to the CV 95% CI from §8. **INSIDE** means the model's performance on the locked 12-month window matches what cross-validation predicted — the planner can tell the legislature: *"The model was evaluated on data it never saw during development, and it performed as expected."* **ABOVE** means the model underperformed on the test window — the planner should flag this honestly. **BELOW** means the model overperformed — unusual but not alarming.

**2. The empirical PI coverage.** The planner constructed a 95% prediction interval around each monthly forecast. The coverage check asks: *"did 95% of actual employment values fall inside my band?"* If coverage is near 95%, the planner tells the legislature: *"When I say the forecast is X ± Y, you can trust that band — it held up on the most recent 12 months."* If coverage is well below 95%, the planner must widen the interval before reporting. If near 100%, the interval is too conservative — the planner can tighten it for a more useful range.

**3. The full-series plot.** Shows the entire 80-year history with the forecast and PI band overlaid on the final 12 months. This is the context figure — it tells the legislature *"here is where the forecast sits in the long-run trajectory of US retail employment."*

**4. The zoomed plot (test + 24 months).** This is the figure the planner puts on the poster. Individual monthly markers let the legislature see exactly where the forecast tracks the seasonal cycle — the November/December holiday peaks, the January trough, the spring-to-fall plateau. Points inside the green band passed; any point outside the band is a month the model's uncertainty was too narrow.

**5. The horizon-vs-RMSE curve.** The planner's honest answer to *"how far ahead can I trust this?"* The curve rises from left to right: 1-month forecasts are tight, 12-month forecasts are wide. The table gives exact numbers the planner can quote: *"Our next-month forecast is accurate to within ±X thousand employees; our next-year forecast is accurate to within ±Y."* This is a "Limitations" figure — the legislature respects a planner who says what the model *cannot* do.

**6. The legislative summary.** The printed report block synthesizes all five diagnostics into the format a decision-maker reads: model name, forecast window, accuracy, interval width, reliability, horizon sensitivity, known limitations, and a retraining recommendation. This is the deliverable the planner walks into the hearing with.

> **A question that often comes up here:** *"How often should the planner actually retrain?"* The notebook recommends a four-part policy. **Monthly retraining is operationally ideal** — the BLS publishes new employment numbers every month, the model fits in seconds, and retraining each month means the next-month forecast is always at horizon=1, the tightest point on the RMSE curve. **Quarterly is the practical fallback** if the legislative reporting cycle is quarterly — the active forecast horizon drifts from 1 to 4 months between presentations, still in the well-behaved part of the curve. **Trigger-based retraining** kicks in regardless of cadence when any of three drift signals fire: the running MAE exceeds the CV 95% CI upper bound (model drift), a recession or policy shock occurs (the kind of remainder spike §3.4 showed), or the empirical PI coverage drops below 90% on a rolling 12-month basis (the band is becoming overconfident). Finally, **an annual model review** re-runs the §3.5 ACF on the most recent decade — if lag-1 and lag-12 are no longer dominant, the *feature selection* itself needs revisiting, not just the training data. The legislative summary the planner brings to the hearing prints all four components.

**The planner's final conclusion:** The six outputs together form the complete legislative deliverable. The verdict says the selection process was honest (or flags it if not). The PI coverage says the band is reliable (or flags that it needs widening). The full-series and zoomed plots give the legislature the context and the detail. The horizon curve gives the honest answer to *"how far ahead can I trust this?"* The legislative summary packages all of it into a one-screen report. I walk into the hearing with a forecast, an uncertainty band, a reliability check, a horizon sensitivity curve, named limitations, and a four-part retraining policy. That is the deliverable the workforce planner has been building toward since §1.

---

## 📝 PAUSE-AND-DO Exercise 2 — Ridge Alpha Tuning (10 minutes)

**Task:** Ridge with `alpha=1.0` may be over- or under-regularized for this series. The workforce planner wants to know: does tuning the regularization strength produce a statistically better forecast, or is the unregularized linear model already the right champion?

Sweep `alpha ∈ [0.01, 0.1, 1, 10, 100]` using the same `Pipeline` + `make_reduction` approach from §8. Run walk-forward CV at each alpha and compare the best Ridge CI to the unregularized Linear baseline. If the CIs overlap, Ridge does not earn its place — the simpler model wins.

**Hints:**
- Build `make_reduction(Pipeline([("select_lags", FunctionTransformer(select_lag1_lag12)), ("regressor", Ridge(alpha=a, random_state=RANDOM_SEED))]), window_length=12, strategy="recursive")` for each alpha.
- Collect `MAE_mean` and `CI_halfwidth` in a `pd.DataFrame`.
- Plot alpha on a log x-axis with error bars.
- Compare the best Ridge CI to `summary.loc["Linear [lag1,lag12]"]`.

Type your code in the cell below.

> 💡 **Gemini Prompt:** *"Sweep Ridge alpha over [0.01, 0.1, 1, 10, 100]. At each alpha, create a make_reduction forecaster wrapping a Pipeline that selects lag-1 and lag-12 (FunctionTransformer(select_lag1_lag12)) then fits Ridge(alpha=alpha, random_state=RANDOM_SEED). Run each through the same ExpandingWindowSplitter CV folds using .clone()/.fit()/.predict(). Collect MAE_mean and CI half-width per alpha. Print the table, plot alpha on a log x-axis vs MAE with error bars, and compare the best Ridge CI to the Linear [lag1,lag12] CI from the summary table."*
>
> **After running, verify:**
> - [ ] The table shows five rows (one per alpha) with MAE_mean and CI half-width columns
> - [ ] The plot has alpha on a log-scaled x-axis with error bars
> - [ ] A printed comparison states whether the best Ridge CI overlaps with the Linear CI
> - [ ] All evaluation uses walk-forward CV on training data only

In [ ]:
# YOUR SOLUTION CODE HERE

# Hints:
# alphas = [0.01, 0.1, 1, 10, 100]
# rows_ridge = []
# for a in alphas:
#     ridge_fc = make_reduction(
#         Pipeline([("select_lags", FunctionTransformer(select_lag1_lag12)),
#                   ("regressor", Ridge(alpha=a, random_state=RANDOM_SEED))]),
#         window_length=12, strategy="recursive")
#     fold_maes = []
#     for tr_idx, va_idx in cv.split(y_train):
#         fc = ridge_fc.clone()
#         fc.fit(y_train.iloc[tr_idx])
#         pred = fc.predict(fh=np.arange(1, len(va_idx) + 1))
#         fold_maes.append(mean_absolute_error(y_train.iloc[va_idx], pred))
#     rows_ridge.append({"alpha": a, "MAE_mean": np.mean(fold_maes),
#                         "CI_hw": np.std(fold_maes, ddof=1)/np.sqrt(n_folds)*t_crit})
# Then plot with errorbar() on a log-x axis.

## 10. Wrap-Up — What the Workforce Planner Delivers

1. **Forecasting is supervised learning with one structural rule: never let the future leak into the past.** That single rule changes the train/test split (recent slice held out), the cross-validation strategy (`ExpandingWindowSplitter` from `sktime`), and what counts as a feature (lags, not random shuffling).
2. **The Week-1 analytics workflow ports cleanly to time series.** EDA → split → baselines → linear features → regularization is the same recipe; only the partition strategy and feature engineering change.
3. **Naive baselines are surprisingly hard to beat.** If your fancy model does not beat seasonal-naive on identical CV folds with non-overlapping CIs, you do not have a champion — you have noise.
4. **The cost of lag features is the loss of the earliest rows.** A 12-month seasonal lag costs you the first year of history. Plan for it.
5. **Walk-forward CV is the time-series spine of CV-first evaluation,** exactly like `StratifiedKFold` was the classification spine in nb08–nb14.

**The workforce planner's deliverable** is now a complete package: a point forecast backed by a cross-validated model comparison, a 95% prediction interval with empirical coverage, a horizon-vs-RMSE curve that honestly states how far ahead the forecast is trustworthy, and a coefficient table the legislature can read. That package — not a single number — is what earns the planner's credibility.

### Beyond This Introduction

This notebook is an introduction — enough to build, evaluate, and defend a lag-feature linear forecast on a real business series. A dedicated time-series forecasting course covers substantially more ground:

| Topic | What it adds |
|---|---|
| **ARIMA / SARIMA** | The classical Box-Jenkins approach: differencing for stationarity, ACF/PACF-based order selection, seasonal terms. Still the benchmark in many industries. |
| **Exponential Smoothing (ETS)** | Holt-Winters and state-space models that weight recent observations more heavily than distant ones. The go-to for short-term inventory and demand planning. |
| **Prophet** | Meta's decomposable model with built-in holiday effects and automatic changepoint detection. Popular in e-commerce and retail forecasting. |
| **Multiple Seasonalities** | Series with daily, weekly, *and* annual cycles simultaneously — hourly electricity demand, web traffic, call-center staffing. |
| **Multivariate Forecasting (VAR)** | Using multiple related series (employment, GDP, consumer confidence) to forecast each other. Includes Granger causality — testing whether one series actually *predicts* another. |
| **Deep Learning (RNN / LSTM / Transformer)** | Sequence models that learn non-linear temporal dependencies from very long histories. Practical when you have thousands of series and large compute budgets. |
| **Hierarchical Reconciliation** | Forecasting at store, region, and national level simultaneously and reconciling the numbers to be consistent. Essential for retail chains and government agencies. |
| **Conformal Prediction Intervals** | Distribution-free uncertainty bands that guarantee coverage *without* the Gaussian assumption we used in §9. The fix when your residuals have heavy tails. |

The lag-feature regression you built today is not a toy — it is genuinely competitive on monthly business series with moderate trend and seasonality. The tools above extend the toolkit when the series is longer, more complex, or demands richer uncertainty quantification.

> **A question that often comes up here:** *"Where do RNNs and transformers fit?"* They are alternatives to lag-feature linear models when (a) the series is long enough (thousands of points, not 960), (b) the dependence is highly non-linear, and (c) you can spare an order of magnitude more compute. For business problems with a few decades of monthly history, a well-engineered lag-feature linear regression is almost always the right starting point — and often the right ending point. Deep learning gets the awareness module it deserves in **nb19**.

**Next stop — nb17: Data Communication and Poster Design.** Now that you have a forecast, a defensible CV-based comparison, and a clean test-set ceremony verdict, the question becomes how to **communicate** them: the six principles of data communication, the eleven-section poster architecture for the M4 deliverable, and the data-ink-ratio cleanup that turns a notebook plot into a poster figure.

---

## Participation Assignment Submission Instructions

1. **Complete both PAUSE-AND-DO exercises** (sections after 8 and 9).
2. **Run all cells** (`Runtime → Run all`).
3. **Save with output** (`File → Download → Download .ipynb`).
4. **Submit to Brightspace** as `nb16_time_series_forecasting_<your_lastname>.ipynb`.

**Bibliography**
- Hyndman & Athanasopoulos: *Forecasting: Principles and Practice* (FPP3) — the [free online textbook](https://otexts.com/fpp3/) is the deep dive on every concept above.
- sktime User Guide: `ExpandingWindowSplitter`, `NaiveForecaster`, and `make_reduction`.
- statsmodels: `STL` decomposition and the autocorrelation function.

<center>

# Thank you!

</center>
